# Задание 1

**Итоговые данные:** шесть файлов в папке `results/`, по одному на источник. В `data/processed/` находятся только рабочие каталоги и журналы прогресса.


[Описание решения](#solution-text) · [Код](#solution-code) · [Проверка качества](#quality-check)


<a id="solution-text"></a>
## Текстовая часть решения


## Как запустить

Использовался Python 3.12 и виртуальное окружение `.venv`. Зависимости: `requests`, `beautifulsoup4`, `selenium`; для браузерного сбора нужен установленный Chrome.

Для проверки готовых данных достаточно выполнить две начальные кодовые ячейки — импорты и настройки — и последнюю ячейку «Проверка качества результатов». Она читает `results/*.jsonl` и записывает `quality_report.json` без обращения к сайтам.

Для нового сбора после общих ячеек выполняются ячейки выбранного сайта по порядку. При запросе ручной проверки нужно пройти её в открытом Chrome и нажать Enter в ноутбуке. Каталоги и журналы в `data/processed/` позволяют продолжать сохранённый сбор.

**Состав решения:** итоговые JSONL, код и журналы прогресса.


## 1.1. Извлечение данных

### Результаты проверки качества

Проверены **6 итоговых файлов, 7 832 записи**. По источникам: IRecommend — 394; LiveJournal — 399; Дзен — 47; 4PDA — 5539; Отзовик — 1356; DRIVE2 — 97.

- Дубли по ID и точному URL внутри каждого набора не обнаружены.
- Пропусков в проверяемых полях ID, ссылки, даты и автора либо издателя нет. Для LiveJournal и Дзена проверяется издатель; отдельный автор текста может быть неизвестен.
- **7 820 записей содержат непустой текст.** Ещё **12 сюжетов Дзена** содержат только сводку и метаданные: полный текст в самом Дзене недоступен, внешние статьи не собирались. Эти записи учтены отдельно.
- Ошибок чтения JSONL и записанных ошибок загрузки нет. У 4PDA поле `fetch_status` отсутствует, поэтому его статус обозначен как `not_recorded`, а не `ok`.

Машиночитаемый отчёт — `quality_report.json`. Проверка подтверждает перечисленные свойства сохранённых данных, но не доказывает полноту архива сайтов или правильность каждого значения.


### 1.1.1 IRecommend

Источник: https://irecommend.ru/content/sait-ozontravel-0

Сайт с отзывами. Один отзыв заключен в div "item-fisrt", "item", "item-odd". Поэтому будем выбирать по атрибутам data-nid, data-product-id и наличию ссылки на отзыв.


Из div можно извлечь:
- Заголовок — .reviewTitle.
- Ссылку на полный отзыв — атрибут href у a.reviewTextSnippet.
- Автора — .authorName, здесь Andante.
- Ссылку на профиль — .authorName a.
- Дату публикации — .created, здесь 03.04.2025.
- Оценку — число заполненных звёзд .star .on: здесь 1 из 5. Класс fivestarWidgetStatic-5 сам по себе не означает оценку 5.
- Количество комментариев — .comments, здесь 39.
- Количество фотографий — data-photos-count, здесь 8.
- Идентификаторы — data-nid и data-product-id, пригодятся для связывания записей и удаления дублей.


**Структура `results/irecommend.jsonl`:** одна строка JSONL — один отзыв.
Отсутствующие значения — `null`, списки — массивы JSON.

- Источник и ссылка: `source`, `product_id`, `review_id`, `url`, `listing_url`.
- Карточка: `title`, `author`, `author_url`, `published_date_raw`, `rating`, `comments_count`, `photos_count`.
- Полный отзыв: `full_title`, `full_author`, `full_author_url`, `published_at`, `published_date_display`, `full_rating`, `text`.
- Дополнительно: `advantages`, `disadvantages`, `usage_experience`, `verdict`, `image_urls`.
- Служебные поля: `attempt_errors`, `collected_at`, `fetch_duration_seconds`, `fetch_error`, `fetch_method`, `fetch_status`, `full_collected_at`.


### Полные тексты отзывов

Основной текст находится в `.reviewBlock [itemprop="reviewBody"]`. Он содержит не только абзацы `<p>`, но и текст между ними, поэтому извлекаем весь текст контейнера. Дополнительно сохраняем дату из микроразметки (с часовым поясом) и отображаемую дату отдельно, оценку, достоинства, недостатки, опыт использования, рекомендацию автора и ссылки на изображения. Тексты комментариев не собираем.

Полные записи сохраняются отдельно в `results/irecommend.jsonl`, при сборе HTML кэшируется в `data/raw/irecommend/reviews/`. Этот кэш после работы удалён. После каждой попытки сохраняем прогресс. Повторный запуск пропускает успешные записи и повторяет неуспешные. 

Сбор изменён на гибридный: сначала используем сохранённый корректный HTML, затем requests, и только при ошибке загрузки или извлечения — Selenium. Браузер создаётся по необходимости и использует eager-загрузку. Для каждой записи фиксируются способ, длительность и ошибки попыток; успешные записи по-прежнему пропускаются.


### 1.1.2 LiveJournal

Источник: https://5koleso.livejournal.com/ — официальный блог журнала «5 Колесо».

При сборе HTML кэшируется в `data/raw/livejournal/` (после работы удалён); каталог ссылок — `data/processed/livejournal_listing.json`; полные записи — `results/livejournal.jsonl`; журнал обхода — `livejournal_crawl.json`. Пауза между запросами — 1 секунда; после трёх ошибок постов подряд остановка. Ошибка страницы ленты означает неполный обход и отмечается в журнале. Сначала используем requests, который уже успешно загрузил ленту и образец поста.

**Структура `results/livejournal.jsonl`:** одна строка JSONL — один пост.
Отсутствующие значения — `null`, списки — массивы JSON.

- Источник и ссылка: `source`, `publisher`, `post_id`, `url`, `listing_url`.
- Публикация: `title`, `full_title`, `text`, `text_author`.
- Дата: `published_date`, `listing_date_display`, `published_datetime_raw`, `published_date_display`.
- Дополнительно: `tags`, `image_urls`, `external_links`.
- Служебные поля: `collected_at`, `fetch_error`, `fetch_method`, `fetch_status`.


### 1.1.3 Дзен
Источник: https://dzen.ru/news/rubric/computers — новостной агрегатор, рубрика «Технологии».

Учитываем только информацию, доступную в самом Дзене. 

Первая ячейка собирает карточки рубрики, включая компактные ссылки, и сохраняет снимки HTML и журнал прокрутки. Вторая получает данные соответствующих сюжетов по каноническому ID. Полный текст сохраняется, если он доступен в Дзене. Если вместо него есть только сводка с цитатами, сохраняем её отдельно в `story_snippets`, а поле `text` оставляем пустым. Сводку не выдаём за полный текст статьи.

Статус `fetch_status=ok` означает успешный сбор страницы сюжета; доступность текста отражается отдельно в `text_status`: `full_text_available`, `summary_only` или `metadata_only`. Отсутствие полного текста в агрегаторе само по себе не является технической ошибкой. Тексты комментариев не собираем.

**Структура `results/dzen.jsonl`:** одна строка JSONL — один сюжет.
Отсутствующие значения — `null`, списки — массивы JSON.

- Источник и ссылки: `source`, `rubric`, `story_id`, `story_url`, `listing_url`.
- Карточка: `title`, `publisher`, `published_date_raw`, `description`.
- Содержимое: `resolved_story_title`, `text`, `text_origin`, `text_status`, `published_at`.
- Источники: `selected_publisher`, `article_url`, `visible_sources`, `story_snippets`, `all_sources_url`.
- Границы сбора: `collection_scope`, `collection_note`.
- Служебные поля: `collected_at`, `duration_seconds`, `enrichment_errors`, `fetch_error`, `fetch_method`, `fetch_status`, `first_seen_run`, `full_collected_at`, `listing_snapshot`, `story_fetch_method`.

`visible_sources` — массив объектов `{url, publisher}`, `story_snippets` — `{text, publisher, url}`. У `summary_only` поле `text=null`: сводка не заменяет полный текст.


Сохранены **47 уникальных сюжетов** из снимков рубрики. У **35** доступен текст в самом Дзене (`full_text_available`), у **12** — только сводка с цитатами и метаданные (`summary_only`). Внешние статьи СМИ не учитываются; ссылки на них остаются метаданными.

Все 47 страниц сюжетов обработаны успешно, дублей ID нет. У 12 сюжетов поле `text` намеренно пустое: это явно отмеченное ограничение полноты текста, а не техническая ошибка загрузки. Основной файл — `results/dzen.jsonl`. Снимки отражают доступную рубрику на момент сбора, а не весь исторический архив новостей.


### 1.1.4 4PDA

Источник: https://4pda.to/forum/index.php?showforum=320 — форум, раздел «Безопасность».

**Выборка:** первые 30 тем первой страницы раздела; первая и 10 последних найденных страниц каждой темы (пересечения исключаются; в коротких темах — все страницы). Это ограниченная выборка описаний и обсуждений, а не выгрузка всего многолетнего форума. Страницы раздела и страниц сообщений различаются; их смещения берём из ссылок сайта.

**Почему такая выборка:** первая страница содержит начало темы и часто описание продукта или инструкции; десять последних страниц дают контекст последних сообщений и позволяют не ограничиваться одной неполной страницей. Это последние сообщения конкретной темы, а не обязательно свежие по календарной дате. Ограничение первыми 30 темами и 10 последними страницами принято для ограничения времени сбора и объёма результата. Середина длинных обсуждений и остальные темы не охвачены, поэтому выборка не отражает весь форум и все мнения пользователей.

Основной результат — `results/4pda.jsonl`; каталог — `4pda_topics.jsonl`; границы и журнал страниц — `4pda_collection.json`; при сборе HTML кэшируется в `data/raw/4pda/` (кэш после работы удалён). Успешные страницы при повторном запуске пропускаются.

**Структура `results/4pda.jsonl`:** одна строка JSONL — один пост.
Отсутствующие значения — `null`, списки — массивы JSON.

- Тема: `source`, `forum_id`, `topic_id`, `topic_title`.
- Сообщение и ссылка: `post_id`, `post_id_source`, `post_key`, `url`, `page_url`.
- Автор и текст: `author`, `author_url`, `published_date_raw`, `text`.
- Дополнительно: `quoted_texts`, `quotes_included_in_text`, `image_urls`, `attachment_urls`.
- Служебные поля: `collected_at`.

Каталог `4pda_topics.jsonl`: `source`, `forum_id`, `topic_id`, `title`, `url`, `description`, `created_date_raw`, `starter`, `starter_url`, `replies_count`, `views_count`, `last_page_offset_seen`.


### 1.1.5 Отзовик

Источник: https://otzovik.com/money_systems/banks/ — каталог банков и банковских услуг.

**Выборка:** обходим найденную навигацию каталога (до 50 страниц). Сохраняем только объекты с **10 и более отзывами**, остальные не включаем в JSONL. Число отзывов берём из счётчика карточки. Для выбранных объектов собираем **до 30 последних отзывов по дате**, а не по полезности; ограничение задаётся в начальной ячейке настроек.

**Почему такие ограничения:** порог 10 отзывов позволяет сосредоточиться на объектах с несколькими описаниями опыта; малообсуждаемые и новые объекты при этом исключены. До 30 последних отзывов на объект ограничивает влияние очень популярных банков на объём выгрузки и показывает последние доступные впечатления. Числа 10 и 30 выбраны как практические ограничения объёма, а не как статистически обоснованные пороги. Такая выборка не показывает полную историю репутации.

Первые две ячейки раздела ниже собирают только каталог и сохраняют HTML одного подходящего объекта для настройки разбора отзывов. Две последние ячейки раздела собирают полные отзывы с сортировкой `order=date_desc` («Сначала новые»). Проверяем выбранный режим сортировки и убывание дат; при неизвестной разметке или неполной пагинации останавливаемся с сохранением прогресса. requests используется первым, при ошибке продолжаем в одной сессии Chrome. При необходимости проверку сайта проходим вручную один раз. При повторной блокировке останавливаемся и сохраняем прогресс.

**Структура `otzovik_objects.jsonl`:** одна строка — подходящий объект каталога: `source`, `object_id`, `object_url`, `name`, `reviews_count`, `category_url`, `listing_url`, `collected_at`. `reviews_count` — целое число. Объекты с неизвестным счётчиком не считаем подходящими автоматически: ошибка разбора останавливает сбор.

При сборе HTML кэшируется в `data/raw/otzovik/` (после работы удалён); итог обхода и счётчики — `data/processed/otzovik_catalog_progress.json`.

**Каталог по результатам запуска:** все 50 страниц, 395 объектов с 10+ отзывами. Порядок обхода страниц определяется ссылками сайта, поэтому номера в выводе могут идти не подряд.

**Структура `results/otzovik.jsonl`:** одна строка — отзыв.

- Объект: `source`, `object_id`, `object_name`, `object_url`.
- Карточка отзыва: `review_id`, `url`, `title`, `author`, `author_url`, `published_at`, `published_date_display`, `rating`, `teaser`, `advantages`, `disadvantages`, `comments_count`, `helpful_count`, `listing_url`.
- Полная страница: `text`, `full_title`, `full_author`, `full_author_url`, `full_published_at`, `full_rating`, `full_advantages`, `full_disadvantages`, `image_urls`.
- Результат загрузки: `fetch_status` (`ok`/`error`), `fetch_error`, `collected_at`. При ошибке поля полного текста могут отсутствовать; `teaser` не считается полным текстом.

`full_*` — значения с полной страницы, исходные поля карточки не затираем; отсутствующие значения — `null`. Тексты комментариев не собираем, только их число. Прогресс по объектам — `otzovik_reviews_progress.json`; успешные объекты и тексты при повторном запуске пропускаются. Выборка фиксируется на момент первого успешного сбора, обновление уже готового объекта требует сброса его отметки в журнале.

**Фактическое покрытие:** каталог из 50 страниц пройден полностью: найдено 1 473 объекта, из них 395 имеют не менее 10 отзывов. Полные отзывы сохранены для **57 объектов**, всего **1 356 отзывов**. Для **56 объектов** выбранная порция отзывов собрана полностью; ещё один объект собран частично, в журнале осталась отметка `running` на момент остановки.

Сбор остановлен вручную из-за времени выполнения и уже полученного объёма данных. Это неполная последовательная выборка объектов каталога, не случайная и не репрезентативная. Нулевое число ошибок в сохранённых отзывах не означает завершения сбора всех 395 объектов. Журнал позволяет продолжить работу по отдельному решению; при подготовке текущего результата сбор не возобновлялся.


### 1.1.6 DRIVE2

https://www.drive2.ru/featured-topics/CVQ0gEAAcHQ/

**Границы:** до 100 публикаций и 10 страниц подборки. Найдены 97 уникальных публикаций на пяти страницах. Это доступная навигация конкретной подборки; полнота исторического архива всей платформы не заявляется. Порядок подборки сохраняем, считать его строго хронологическим без проверки нельзя.

**Структура `results/drive2.jsonl`:** одна строка — публикация.

- Источник и ссылка: `source`, `topic`, `topic_url`, `post_id`, `post_type`, `url`, `listing_url`.
- Карточка: `title`, `author`, `author_url`, `listing_date_display`, `teaser`, `likes_count`, `comments_count`, `collected_at`.
- Полная публикация: `full_title`, `text`, `published_at`, `published_date_display`, `modified_at`, `text_author`, `image_urls`, `body_links`, `video_urls`, `full_likes_count`, `full_comments_count`.
- Сбор: `fetch_status`, `fetch_error`, `fetch_method`, `full_collected_at`.

`post_type=b` — блог, `post_type=l` — запись бортжурнала по виду ссылки. `text_author=null`, если отдельное авторство не определено. Списки ссылок — массивы строк, недоступные значения — `null`. Тексты комментариев, изображения и видео как файлы не скачиваем; сохраняем счётчики и доступные ссылки. Ссылки из текста не обходим.

Каталог — `drive2_listing.jsonl`, журнал границ/полноты — `drive2_collection.json`, при сборе HTML кэшируется в `data/raw/drive2/` (кэш после работы удалён). После ошибки сбор останавливается; готовые тексты при повторном запуске пропускаются.


## Проверка качества итоговых данных

Отдельная последняя ячейка читает все `results/*.jsonl`. Проверяет число записей, дубли непустых ID и точных URL, пропуски ID/URL/автора или издателя/даты/текста, статусы и причины ошибок загрузки, ошибки чтения JSONL. Пропуск — отсутствие ключа, `null` или пустая строка. Доля считается среди корректно прочитанных записей.

У LiveJournal и Дзена проверяется **издатель**, а не неизвестный автор текста. У 4PDA отсутствие `fetch_status` показывается как `not_recorded`, не как `ok`. Записи Дзена `summary_only`/`metadata_only` учитываются отдельно: отсутствие полного текста не скрывается, но и не считается неожиданной ошибкой. Проверка наличия даты не проверяет правильность самой даты; диапазоны дат здесь не рассчитываются. Дубли URL считаются по точному совпадению; одинаковые ID/URL — сигнал для проверки, не автоматическое удаление записей.

Отчёт сохраняется в `quality_report.json` в корне проекта, чтобы в `results/` оставались только шесть итоговых датасетов. Проверка не оценивает полноту исторического архива или правильность каждого извлечённого значения.


### Как можно улучшить мое решение

Думаю стоит об этом написать, так как понимаю, что можно сделать лучше, но не знаю как и вообще можно ли это сделать, но так подсказывает моя интуицая.

Думаю следует сделать базу данных, однако не знаю насколько можно объединить информацию с нескольких источников в одну (возможно, после разбиения источников по классам, это будет возможно) в случае задачи с большем количеством источников как осуждается в задании 2.
Однако для этой задачи может получится сделать ключевые поля одинаковые для всех источников.

## 1.2. Классификация источников

Разделим источники на два класса по содержанию:

1. **Отзывы и пользовательский опыт:** IRecommend, Отзовик, 4PDA.
2. **Новости/блоги:** «5 Колесо» в LiveJournal, DRIVE2, Дзен.

Можно разделить более подробно, например 4PDA - форум, Дзен - новостной агрегатор и тд. Однако подразумаю, что для задачи именно такого разделения достаточно, по крайней мере с этими источниками.
Мы разделяем пользовательсикий опыт от новостей.

### Ограничения проверки и дальнейшая доработка

Даты пока сохранены в исходных полях: ISO-формат, календарная дата или строка сайта, например `27.07.10, 06:53` у 4PDA. Единого нормализованного поля даты пока нет. При дальнейшей обработке нужно сохранять исходное значение, отдельно приводить распознанную дату к общему формату и явно отмечать неизвестный часовой пояс; относительные даты следует интерпретировать с учётом момента сбора.

Аналогично, поля карточки и полной публикации пока сохранены раздельно (`title`/`full_title`, `published_at`/`full_published_at`). Для аналитического использования можно добавить единые выбранные значения с указанными правилами приоритета и признаками конфликтов. Исходные значения при этом должны оставаться доступными.


<a id="solution-code"></a>
## Код

Сначала запускаем импорты и настройки, затем ячейки нужного сайта по порядку.

[Импорты и настройки](#setup) · [IRecommend](#code-source-1) · [LiveJournal](#code-source-2) · [Дзен](#code-source-3) · [4PDA](#code-source-4) · [Отзовик](#code-source-5) · [DRIVE2](#code-source-6)


<a id="setup"></a>
### Импорты и настройки



In [42]:
import hashlib
import json
import re
import time
from collections import Counter, deque
from datetime import datetime, timezone
from pathlib import Path
from time import monotonic
from urllib.parse import parse_qs, parse_qsl, urlencode, urljoin, urlsplit, urlunsplit

import requests
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.common.exceptions import TimeoutException, WebDriverException
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait


def get_text(element):
    return element.get_text(' ', strip=True) if element is not None else None


def get_number(value):
    compact = ''.join((value or '').split())
    return int(compact) if compact.isdecimal() else None


def read_jsonl(path):
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]


def save_jsonl(path, records):
    temporary = path.with_suffix('.jsonl.tmp')
    with temporary.open('w', encoding='utf-8') as file:
        for record in records:
            file.write(json.dumps(record, ensure_ascii=False) + '\n')
    temporary.replace(path)


In [43]:
PROJECT = Path.cwd()
RAW_DIR = PROJECT / 'data/raw'
WORK_DIR = PROJECT / 'data/processed'  # Каталоги и журналы прогресса.
FINAL_DIR = PROJECT / 'results'  # Только шесть итоговых наборов данных.
REQUEST_DELAY = 1

# IRecommend: только отзывы об Ozon.travel
IRECOMMEND_URL = 'https://irecommend.ru/content/sait-ozontravel-0'
IRECOMMEND_PRODUCT_ID = '2308896'
IRECOMMEND_PAGE_LIMIT = 1000
IRECOMMEND_REVIEW_LIMIT = None

# LiveJournal: найденная навигация одного блога
LIVEJOURNAL_URL = 'https://5koleso.livejournal.com/'
LIVEJOURNAL_PAGE_LIMIT = 1000

# Дзен: снимок рубрики, а не весь архив
DZEN_URL = 'https://dzen.ru/news/rubric/computers'
DZEN_STORY_TARGET = 200
DZEN_SCROLL_LIMIT = 30
DZEN_VISIBLE_BROWSER = False
DZEN_PROCESS_LIMIT = None
DZEN_REPARSE = False  # True — повторно разобрать уже успешные сюжеты.

# 4PDA: 30 тем, первая и 10 последних страниц каждой темы
FORUM_URL = 'https://4pda.to/forum/index.php'
FORUM_ID = '320'
TOPICS_LIMIT = 30
LAST_PAGES = 10
POSTS_PER_PAGE = 20

# Отзовик: весь каталог, объекты минимум с 10 отзывами, до 30 свежих отзывов.
OTZOVIK_URL = 'https://otzovik.com/money_systems/banks/'
OTZOVIK_CATALOG_LIMIT = 50
OTZOVIK_MIN_REVIEWS = 10
OTZOVIK_REVIEWS_LIMIT = 30

# DRIVE2: подборка «Безопасность», до 100 публикаций по найденной навигации.
DRIVE2_URL = 'https://www.drive2.ru/featured-topics/CVQ0gEAAcHQ/'
DRIVE2_POST_LIMIT = 100
DRIVE2_PAGE_LIMIT = 10

WORK_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)
for name in ['irecommend', 'livejournal', 'dzen', '4pda', 'otzovik', 'drive2']:
    (RAW_DIR / name).mkdir(parents=True, exist_ok=True)


<a id="code-source-1"></a>
### IRecommend

[К текстовой части](#solution-text)


In [ ]:
# 1. Обходим страницы Ozon.travel и сохраняем карточки отзывов.


url = IRECOMMEND_URL
product_id = IRECOMMEND_PRODUCT_ID
base = urlsplit(url)
raw_dir = RAW_DIR / "irecommend"
output_dir = WORK_DIR
raw_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)


# Обходим только пагинацию исходного объекта, не рекомендации других товаров.
pending = deque([0])
scheduled = {0}
page_log = []
reviews_by_url = {}
max_pages = IRECOMMEND_PAGE_LIMIT

with requests.Session() as session:
    while pending:
        if len(page_log) >= max_pages:
            raise RuntimeError("Достигнут лимит страниц; сбор не завершён.")

        page_number = pending.popleft()
        page_url = url if page_number == 0 else f"{url}?{urlencode({'page': page_number})}"
        if page_log:
            time.sleep(1)

        response = session.get(page_url, timeout=30)
        response.raise_for_status()
        html = response.content
        (raw_dir / f"page_{page_number:03d}.html").write_bytes(html)
        soup = BeautifulSoup(html, "html.parser")

        actual = urlsplit(response.url)
        actual_page = parse_qs(actual.query).get("page", ["0"])[0]
        if (actual.hostname != base.hostname
                or actual.path.rstrip("/") != base.path.rstrip("/")
                or actual_page != str(page_number)):
            raise RuntimeError(f"Неожиданный адрес после запроса: {response.url}")

        cards = soup.select(
            f'div[data-nid][data-product-id="{product_id}"]'
        )
        cards = [c for c in cards if c.select_one("a.reviewTextSnippet[href]")]
        if not cards:
            raise RuntimeError(
                f"Нет карточек Ozon.travel на {page_url}. "
                "Проверьте сохранённый HTML: возможна заглушка или изменение разметки."
            )

        collected_at = datetime.now(timezone.utc).isoformat()
        new_count = 0
        for card in cards:
            link = card.select_one("a.reviewTextSnippet[href]")
            review_url = urljoin(response.url, link["href"])
            if review_url in reviews_by_url:
                continue
            author = card.select_one(".authorName a")
            stars = card.select_one(".starsRating .fivestarWidgetStatic")
            reviews_by_url[review_url] = {
                "source": "irecommend",
                "product_id": product_id,
                "review_id": card.get("data-nid"),
                "url": review_url,
                "title": get_text(card.select_one(".reviewTitle")),
                "author": get_text(author),
                "author_url": urljoin(response.url, author["href"])
                    if author is not None and author.get("href") else None,
                "published_date_raw": get_text(card.select_one(".created")),
                "rating": len(stars.select(".star .on"))
                    if stars is not None else None,
                "comments_count": get_number(get_text(card.select_one(".comments"))),
                "photos_count": get_number(card.get("data-photos-count")),
                "listing_url": page_url,
                "collected_at": collected_at,
            }
            new_count += 1

        # Берём номера страниц из реальных ссылок, а не из числа голосов.
        for anchor in soup.select("a[href]"):
            target = urlsplit(urljoin(response.url, anchor["href"]))
            if (target.hostname != base.hostname
                    or target.path.rstrip("/") != base.path.rstrip("/")):
                continue
            values = parse_qs(target.query).get("page", [])
            if len(values) != 1 or not values[0].isascii() or not values[0].isdigit():
                continue
            number = int(values[0])
            if number not in scheduled:
                scheduled.add(number)
                pending.append(number)

        page_log.append({
            "url": page_url,
            "cards_count": len(cards),
            "new_reviews_count": new_count,
            "collected_at": collected_at,
        })
        # Промежуточное сохранение: данные останутся и при сбое следующей страницы.
        reviews = list(reviews_by_url.values())
        with (output_dir / "irecommend_reviews.jsonl").open("w", encoding="utf-8") as f:
            for review in reviews:
                f.write(json.dumps(review, ensure_ascii=False) + "\n")
        (output_dir / "irecommend_pages.json").write_text(
            json.dumps(page_log, ensure_ascii=False, indent=2), encoding="utf-8"
        )
        print(f"Страница {page_number + 1}: {len(cards)} карточек, "
              f"новых {new_count}, всего {len(reviews)}")

print(f"Обход найденных ссылок завершён: {len(page_log)} страниц, {len(reviews)} отзывов.")


In [ ]:
# 2. Функция извлечения полного отзыва из HTML.


def parse_irecommend_review(html, review_url):
    page = BeautifulSoup(html, "html.parser")
    # Проверяем, что это нужный отзыв, а не рекомендация на странице.
    block = None
    for candidate in page.select('div.reviewBlock[itemprop="review"]'):
        title_link = candidate.select_one(".reviewTitle a[href]")
        if title_link and urlsplit(urljoin(review_url, title_link["href"])).path.rstrip("/") == urlsplit(review_url).path.rstrip("/"):
            block = candidate
            break
    if block is None:
        raise ValueError("Не найден блок нужного отзыва; проверьте HTML")

    body = block.select_one('[itemprop="reviewBody"]')
    if body is None:
        raise ValueError("Не найден основной текст отзыва")

    # Изображения берём из полного отзыва, а не из миниатюр списка.
    image_urls = []
    for image in body.select('a[itemprop="contentUrl"]'):
        image_url = image.get("data-src") or image.get("href")
        if image_url:
            image_url = urljoin(review_url, image_url)
            if image_url not in image_urls:
                image_urls.append(image_url)

    # Копия для очистки: текст бывает и в <p>, и между тегами абзацев.
    clean_body = BeautifulSoup(str(body), "html.parser")
    for element in clean_body.select("script, style, noscript, svg"):
        element.decompose()
    full_text = clean_body.get_text("\n", strip=True)
    if not full_text:
        raise ValueError("Основной текст пустой")

    def text(selector):
        element = block.select_one(selector)
        return element.get_text(" ", strip=True) if element is not None else None

    def content(selector):
        element = block.select_one(selector)
        return element.get("content") if element is not None else None

    def notes(selector):
        container = block.select_one(selector)
        if container is None:
            return None
        return [e.get_text(" ", strip=True)
                for e in container.select('li [itemprop="name"]')]

    author = block.select_one('[itemprop="author"] a[href]')
    experience = None
    for item in block.select(".extraInfo li"):
        label = item.select_one(".label")
        value = item.select_one(".item-data")
        if label and value and "Опыт использования" in label.get_text():
            experience = value.get_text(" ", strip=True)

    rating = content('[itemprop="ratingValue"]')
    return {
        "full_title": text(".reviewTitle"),
        "full_author": text('[itemprop="author"] [itemprop="name"]'),
        "full_author_url": urljoin(review_url, author["href"]) if author else None,
        "published_at": content('[itemprop="datePublished"]'),
        "published_date_display": text(".com-date"),
        "full_rating": int(rating) if rating and rating.isdecimal() else None,
        "text": full_text,
        "advantages": notes('[itemprop="positiveNotes"]'),
        "disadvantages": notes('[itemprop="negativeNotes"]'),
        "usage_experience": experience,
        "verdict": text(".conclusion .verdict"),
        "image_urls": image_urls,
    }


In [ ]:
# 3. Получаем полные отзывы: сохранённый HTML → requests → браузер.
output_dir = WORK_DIR
raw_reviews_dir = RAW_DIR / "irecommend/reviews"
raw_reviews_dir.mkdir(parents=True, exist_ok=True)
source_path = output_dir / "irecommend_reviews.jsonl"
full_path = FINAL_DIR / "irecommend.jsonl"


records = read_jsonl(source_path)
completed = {r["url"]: r for r in read_jsonl(full_path)}


def save_irecommend_progress():
    # Заменяем файл только после полной записи временного файла.
    temporary = full_path.with_suffix(".jsonl.tmp")
    with temporary.open("w", encoding="utf-8") as file:
        for record in records:
            if record["url"] in completed:
                file.write(json.dumps(completed[record["url"]], ensure_ascii=False) + "\n")
    temporary.replace(full_path)


# Гибридный сбор: сохранённый HTML → requests → Selenium при ошибке.

max_to_process = IRECOMMEND_REVIEW_LIMIT
request_delay = REQUEST_DELAY
processed_now = 0
consecutive_errors = 0
remaining = [r for r in records if not (
    completed.get(r["url"], {}).get("fetch_status") == "ok"
    and completed.get(r["url"], {}).get("text")
)]
print(f"Осталось обработать: {len(remaining)}")
browser = None


def check_review_address(actual_url, expected_url):
    actual, expected = urlsplit(actual_url), urlsplit(expected_url)
    if (actual.hostname != expected.hostname
            or actual.path.rstrip("/") != expected.path.rstrip("/")):
        raise ValueError(f"Неожиданный переход: {actual_url}")


try:
    with requests.Session() as session:
        for index, record in enumerate(remaining, 1):
            if max_to_process is not None and processed_now >= max_to_process:
                break
            if processed_now:
                time.sleep(request_delay)

            started = monotonic()
            result = dict(record)
            attempts = []
            parsed = None
            method = None
            file_id = hashlib.sha256(record["url"].encode()).hexdigest()[:20]
            html_path = raw_reviews_dir / f"{file_id}.html"

            # Если HTML уже скачан, повторная сеть не нужна.
            if html_path.exists():
                try:
                    parsed = parse_irecommend_review(html_path.read_bytes(), record["url"])
                    method = "cache"
                except ValueError as error:
                    attempts.append({"method": "cache", "error": str(error)})

            if parsed is None:
                try:
                    response = session.get(record["url"], timeout=(5, 10))
                    response.raise_for_status()
                    check_review_address(response.url, record["url"])
                    parsed = parse_irecommend_review(response.content, record["url"])
                    html_path.write_bytes(response.content)
                    method = "requests"
                except (requests.RequestException, ValueError) as error:
                    attempts.append({"method": "requests", "error": str(error)})

            if parsed is None:
                try:
                    # Браузер создаём только при первом неудачном requests.
                    if browser is None:
                        options = webdriver.ChromeOptions()
                        options.page_load_strategy = "eager"
                        browser = webdriver.Chrome(options=options)
                        browser.set_page_load_timeout(15)
                    try:
                        browser.get(record["url"])
                    except TimeoutException:
                        # Таймаут навигации ещё не означает отсутствие текста.
                        # Пробуем прочитать уже загруженный DOM.
                        browser.execute_script("window.stop();")
                    WebDriverWait(browser, 8).until(
                        lambda b: b.find_elements(
                            By.CSS_SELECTOR, '.reviewBlock [itemprop="reviewBody"]'
                        )
                    )
                    check_review_address(browser.current_url, record["url"])
                    html = browser.page_source
                    parsed = parse_irecommend_review(html, record["url"])
                    html_path.write_text(html, encoding="utf-8")
                    method = "selenium"
                except (WebDriverException, ValueError) as error:
                    attempts.append({"method": "selenium", "error": str(error)})
                    # Сломанный драйвер не используем для следующей записи.
                    if browser is not None:
                        try:
                            browser.quit()
                        except WebDriverException:
                            pass
                        browser = None

            if parsed is not None:
                result.update(parsed)
                result.update({"fetch_status": "ok", "fetch_error": None,
                               "fetch_method": method})
                consecutive_errors = 0
            else:
                result.update({"fetch_status": "error", "fetch_method": None,
                               "fetch_error": attempts[-1]["error"] if attempts else "Нет текста"})
                consecutive_errors += 1
            result["attempt_errors"] = attempts
            result["fetch_duration_seconds"] = round(monotonic() - started, 2)
            result["full_collected_at"] = datetime.now(timezone.utc).isoformat()
            completed[record["url"]] = result
            save_irecommend_progress()
            processed_now += 1
            print(f"{index}/{len(remaining)}: {result['fetch_status']} "
                  f"[{method or 'requests + selenium'}], "
                  f"{result['fetch_duration_seconds']} с — {record['url']}")
            if result["fetch_status"] == "error":
                print("Причина:", result["fetch_error"])
            if consecutive_errors >= 3:
                print("Три ошибки подряд: остановка, прогресс сохранён.")
                break
finally:
    if browser is not None:
        try:
            browser.quit()
        except WebDriverException:
            pass

successful = sum(completed.get(r["url"], {}).get("fetch_status") == "ok" for r in records)
errors = sum(completed.get(r["url"], {}).get("fetch_status") == "error" for r in records)
print(f"Успешно: {successful}; ошибок: {errors}; "
      f"не обработано: {len(records) - successful - errors}; всего: {len(records)}")


<a id="code-source-2"></a>
### LiveJournal

[К текстовой части](#solution-text)


In [ ]:
# 1. Получение HTML ленты и одного поста для проверки разметки.


lj_url = LIVEJOURNAL_URL
lj_raw_dir = RAW_DIR / "livejournal"
lj_raw_dir.mkdir(parents=True, exist_ok=True)
lj_listing_path = lj_raw_dir / "listing_skip_0.html"
if not lj_listing_path.exists():
    lj_response = requests.get(lj_url, timeout=(5, 20))
    lj_response.raise_for_status()
    lj_listing_path.write_bytes(lj_response.content)

lj_soup = BeautifulSoup(lj_listing_path.read_bytes(), "html.parser")
lj_links = lj_soup.select(
    'article.entryunit[data-journal="5koleso"] .entryunit__title a[href]'
)
print("Заголовок ленты:", lj_soup.title.get_text(strip=True) if lj_soup.title else None)
print("Постов основной ленты:", len(lj_links))
if not lj_links:
    raise ValueError("Не найдены посты; проверьте HTML ленты")

lj_first_url = urljoin(lj_url, lj_links[0]["href"])
lj_sample_path = lj_raw_dir / "sample_post.html"
# Проверяем, что прежний образец относится к выбранному посту.
lj_need_sample = True
if lj_sample_path.exists():
    lj_existing = BeautifulSoup(lj_sample_path.read_bytes(), "html.parser")
    lj_canonical = lj_existing.select_one('meta[property="og:url"]')
    lj_need_sample = not (
        lj_canonical and lj_canonical.get("content")
        and urlsplit(lj_canonical["content"]).path == urlsplit(lj_first_url).path
    )
if lj_need_sample:
    lj_response = requests.get(lj_first_url, timeout=(5, 20))
    lj_response.raise_for_status()
    lj_sample_path.write_bytes(lj_response.content)
lj_sample_soup = BeautifulSoup(lj_sample_path.read_bytes(), "html.parser")
print("Пост:", lj_first_url)
print("Заголовок поста:", lj_sample_soup.title.get_text(strip=True) if lj_sample_soup.title else None)
print("Сохранено:", lj_sample_path)


In [ ]:
# LiveJournal: обход ленты и сбор полных постов. Повторный запуск продолжает сбор.


lj_url = LIVEJOURNAL_URL
lj_raw = RAW_DIR / "livejournal"
lj_output = WORK_DIR
lj_raw.mkdir(parents=True, exist_ok=True)
lj_output.mkdir(parents=True, exist_ok=True)
lj_delay = REQUEST_DELAY
lj_max_pages = LIVEJOURNAL_PAGE_LIMIT
lj_posts_file = FINAL_DIR / "livejournal.jsonl"


def lj_save_json(path, data):
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
    temporary.replace(path)


def lj_read_json(path, default):
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else default


def lj_parse_listing(html, listing_url):
    soup = BeautifulSoup(html, "html.parser")
    entries = []
    for article in soup.select('article.entryunit[data-journal="5koleso"]'):
        link = article.select_one(".entryunit__title a[href]")
        if link is None:
            continue
        target = urlsplit(urljoin(listing_url, link["href"]))
        if target.hostname != "5koleso.livejournal.com" or not re.fullmatch(r"/\d+\.html", target.path):
            continue
        date = article.select_one("time")
        entries.append({
            "source": "livejournal", "publisher": "5koleso",
            "post_id": target.path.strip("/").removesuffix(".html"),
            "url": f"https://5koleso.livejournal.com{target.path}",
            "title": link.get_text(" ", strip=True),
            "published_date": date.get("datetime") if date else None,
            "listing_date_display": date.get_text(" ", strip=True) if date else None,
            "listing_url": listing_url,
        })
    if not entries:
        raise ValueError("Не найдены записи основной ленты; проверьте HTML, обход остановлен")
    offsets = set()
    for link in soup.select("a[href]"):
        target = urlsplit(urljoin(listing_url, link["href"]))
        if target.hostname != "5koleso.livejournal.com" or target.path not in ("", "/"):
            continue
        values = parse_qs(target.query).get("skip", [])
        if len(values) == 1 and values[0].isascii() and values[0].isdigit():
            offsets.add(int(values[0]))
    return entries, sorted(offsets)


def lj_parse_post(html, post_url):
    soup = BeautifulSoup(html, "html.parser")
    body = soup.select_one("article.aentry-post .aentry-post__text")
    title = soup.select_one("h1.aentry-post__title")
    if body is None or title is None:
        raise ValueError("Не найдены заголовок и основной текст полного поста")
    canonical = soup.select_one('meta[property="og:url"]')
    if canonical and canonical.get("content"):
        if urlsplit(canonical["content"]).path.rstrip("/") != urlsplit(post_url).path.rstrip("/"):
            raise ValueError("Страница относится к другому посту")
    images = []
    for image in body.select("img"):
        image_url = image.get("data-src") or image.get("data-original") or image.get("src")
        if image_url:
            absolute = urljoin(post_url, image_url)
            if absolute.startswith(("http://", "https://")) and absolute not in images:
                images.append(absolute)
    external_links = []
    for link in body.select('a[href]'):
        target = urljoin(post_url, link['href'])
        host = urlsplit(target).hostname
        if target.startswith(('http://', 'https://')) and host != '5koleso.livejournal.com':
            if target not in external_links:
                external_links.append(target)
    clean = BeautifulSoup(str(body), "html.parser")
    for element in clean.select("script, style, noscript, svg, .lj-like, .ljcut-link"):
        element.decompose()
    text = clean.get_text(" ", strip=True)
    if not text:
        raise ValueError("Пустой текст поста")
    date = soup.select_one(".aentry-head__date time")
    tags = [tag.get_text(" ", strip=True) for tag in soup.select('article.aentry-post a[rel~="tag"]')]
    return {
        "full_title": title.get_text(" ", strip=True),
        "text": text,
        "published_datetime_raw": date.get("datetime") if date else None,
        "published_date_display": date.get_text(" ", strip=True) if date else None,
        "text_author": None,  # Не приравниваем аккаунт журнала к автору статьи.
        "tags": list(dict.fromkeys(tags)) or None,
        "image_urls": images,
        "external_links": external_links,
    }


# Сохранённый образец используем без повторного скачивания.
lj_sample = lj_raw / "sample_post.html"
if lj_sample.exists():
    sample_soup = BeautifulSoup(lj_sample.read_bytes(), "html.parser")
    sample_url = sample_soup.select_one('meta[property="og:url"]')
    if sample_url and sample_url.get("content"):
        sample_target = urlsplit(sample_url["content"])
        if sample_target.hostname == "5koleso.livejournal.com" and re.fullmatch(r"/\d+\.html", sample_target.path):
            sample_cache = lj_raw / f"post_{sample_target.path.strip('/').removesuffix('.html')}.html"
            if not sample_cache.exists():
                sample_cache.write_bytes(lj_sample.read_bytes())

lj_catalog = lj_read_json(lj_output / "livejournal_listing.json", {})
lj_completed = {}
if lj_posts_file.exists():
    for line in lj_posts_file.read_text(encoding="utf-8").splitlines():
        if line.strip():
            item = json.loads(line)
            lj_completed[item["url"]] = item


def lj_save_posts():
    temporary = lj_posts_file.with_suffix(".jsonl.tmp")
    with temporary.open("w", encoding="utf-8") as file:
        for item in lj_completed.values():
            file.write(json.dumps(item, ensure_ascii=False) + "\n")
    temporary.replace(lj_posts_file)


lj_last_request = 0


def lj_fetch(session, page_url, cache_path, parser):
    global lj_last_request
    if cache_path.exists():
        try:
            return parser(cache_path.read_bytes(), page_url), "cache"
        except ValueError:
            pass
    time.sleep(max(0, lj_delay - (time.monotonic() - lj_last_request)))
    lj_last_request = time.monotonic()
    response = session.get(page_url, timeout=(5, 20))
    response.raise_for_status()
    actual, expected = urlsplit(response.url), urlsplit(page_url)
    if actual.hostname != expected.hostname or actual.path.rstrip("/") != expected.path.rstrip("/"):
        raise ValueError(f"Неожиданный переход: {response.url}")
    if expected.path in ("", "/") and parse_qs(actual.query).get("skip", ["0"]) != parse_qs(expected.query).get("skip", ["0"]):
        raise ValueError("Сервер вернул другую страницу ленты")
    parsed = parser(response.content, page_url)
    cache_path.write_bytes(response.content)
    return parsed, "requests"


lj_pending = deque([0])
lj_scheduled = {0}
lj_page_log = []
lj_listing_complete = True
with requests.Session() as lj_session:
    # 1. Собираем каталог ссылок, следуя только навигации этого блога.
    while lj_pending:
        if len(lj_page_log) >= lj_max_pages:
            print("Лимит страниц: обход ленты неполный.")
            lj_listing_complete = False
            break
        skip = lj_pending.popleft()
        listing_url = lj_url if skip == 0 else f"{lj_url}?skip={skip}"
        try:
            (entries, offsets), method = lj_fetch(
                lj_session, listing_url, lj_raw / f"listing_skip_{skip}.html", lj_parse_listing
            )
        except (requests.RequestException, ValueError) as error:
            lj_page_log.append({"url": listing_url, "status": "error", "error": str(error)})
            lj_listing_complete = False
            print("Ошибка ленты:", error)
            break
        for entry in entries:
            lj_catalog.setdefault(entry["url"], entry)
        for offset in offsets:
            if offset > skip and offset not in lj_scheduled:
                lj_scheduled.add(offset)
                lj_pending.append(offset)
        lj_page_log.append({"url": listing_url, "status": "ok", "posts": len(entries), "method": method})
        lj_save_json(lj_output / "livejournal_listing.json", lj_catalog)
        print(f"Лента skip={skip}: {len(entries)} постов, уникальных {len(lj_catalog)} [{method}]")

    lj_save_json(lj_output / "livejournal_crawl.json", {
        "listing_complete": lj_listing_complete, "pages": lj_page_log,
        "collected_at": datetime.now(timezone.utc).isoformat(),
    })
    # 2. Скачиваем полные посты; успехи прошлого запуска пропускаем.
    consecutive_errors = 0
    for index, entry in enumerate(lj_catalog.values(), 1):
        previous = lj_completed.get(entry["url"], {})
        if previous.get("fetch_status") == "ok" and previous.get("text"):
            continue
        result = dict(entry)
        try:
            parsed, method = lj_fetch(
                lj_session, entry["url"], lj_raw / f"post_{entry['post_id']}.html", lj_parse_post
            )
            result.update(parsed)
            result.update(fetch_status="ok", fetch_error=None, fetch_method=method)
            consecutive_errors = 0
        except (requests.RequestException, ValueError) as error:
            result.update(fetch_status="error", fetch_error=str(error))
            consecutive_errors += 1
        result["collected_at"] = datetime.now(timezone.utc).isoformat()
        lj_completed[entry["url"]] = result
        lj_save_posts()
        print(f"Пост {index}/{len(lj_catalog)}: {result['fetch_status']} — {entry['url']}")
        if result["fetch_status"] == "error":
            print("Причина:", result["fetch_error"])
        if consecutive_errors >= 3:
            print("Три ошибки подряд: остановка, прогресс сохранён.")
            break

lj_ok = sum(lj_completed.get(u, {}).get("fetch_status") == "ok" for u in lj_catalog)
lj_errors = sum(lj_completed.get(u, {}).get("fetch_status") == "error" for u in lj_catalog)
print(f"Итог: каталог {len(lj_catalog)}, успешно {lj_ok}, ошибок {lj_errors}, "
      f"не обработано {len(lj_catalog) - lj_ok - lj_errors}.")
print("Обход найденной навигации завершён:", lj_listing_complete)
print("Файл:", lj_posts_file)


<a id="code-source-3"></a>
### Дзен

[К текстовой части](#solution-text)


In [ ]:
# 1. Сбор каталога сюжетов рубрики. Весь код находится в ноутбуке.

def normalize(text):
    return ' '.join((text or '').split())


def soup_text(element):
    if element is None:
        return None
    copied = BeautifulSoup(str(element), 'html.parser')
    for tag in copied.select('script,style,noscript,svg,button'):
        tag.decompose()
    return normalize(copied.get_text(' ', strip=True)) or None


def dzen_parse_cards(html, base_url):
    page = BeautifulSoup(html, 'html.parser')
    scope = page.select_one('#news-page') or page
    cards = {}
    for anchor in scope.select('a[href*="/news/story/"]'):
        story_url = urljoin(base_url, anchor['href'])
        target = urlsplit(story_url)
        if target.hostname != 'dzen.ru' or not target.path.startswith('/news/story/'):
            continue
        if parse_qs(target.query).get('rubric', ['computers'])[0] != 'computers':
            continue
        # Компактная ссылка может быть внутри чужой карточки.
        # Не берём из неё чужой заголовок и автора.
        title_text = soup_text(anchor)
        article = anchor.find_parent('article')
        own_card = None
        if article is not None:
            heading = article.select_one('[data-testid="card-title"]')
            heading_link = heading.find_parent('a') if heading else None
            if heading_link and urlsplit(urljoin(base_url, heading_link.get('href', ''))).path == target.path:
                own_card = article
                title_text = title_text or soup_text(heading)
        if not title_text:
            continue
        publisher = own_card.select_one('[class*="card-author__authorTitle"]') if own_card else None
        date = own_card.select_one('[class*="card-meta__cardMeta"]') if own_card else None
        description = own_card.select_one('[data-testid="card-description"]') if own_card else None
        story_id = target.path.rstrip('/').split('/')[-1]
        candidate = {'source': 'dzen_news', 'rubric': 'computers',
                     'story_id': story_id, 'story_url': story_url, 'title': title_text,
                     'publisher': soup_text(publisher), 'published_date_raw': soup_text(date),
                     'description': soup_text(description)}
        if story_id not in cards:
            cards[story_id] = candidate
        else:
            for key, value in candidate.items():
                if value and not cards[story_id].get(key):
                    cards[story_id][key] = value
    return page, list(cards.values())


def collect_dzen_listing():
    project = PROJECT
    headed = DZEN_VISIBLE_BROWSER
    max_stories = DZEN_STORY_TARGET
    max_scrolls = DZEN_SCROLL_LIMIT
    # Собираем только доступные карточки рубрики.
    raw = project / 'data/raw/dzen'
    output = project / 'data/processed'
    raw.mkdir(parents=True, exist_ok=True)
    output.mkdir(parents=True, exist_ok=True)
    path = output / 'dzen_stories_listing.jsonl'
    catalog = {r['story_id']: r for r in read_jsonl(path)}
    run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%S%fZ')
    snapshots = raw / 'listing_runs' / run_id
    snapshots.mkdir(parents=True, exist_ok=True)
    options = webdriver.ChromeOptions()
    options.page_load_strategy = 'eager'
    if not headed:
        options.add_argument('--headless=new')
    browser = webdriver.Chrome(options=options)
    browser.set_page_load_timeout(25)
    seen = set()
    steps = []
    idle = 0
    reason = 'interrupted'
    url = DZEN_URL
    try:
        try:
            browser.get(url)
        except TimeoutException:
            pass
        WebDriverWait(browser, 20).until(
            lambda b: b.find_elements('css selector', 'article a[href*="/news/story/"]'))
        for step in range(max_scrolls + 1):
            actual = urlsplit(browser.current_url)
            if actual.hostname != 'dzen.ru' or actual.path.rstrip('/') != '/news/rubric/computers':
                raise ValueError(f'Неожиданный адрес рубрики: {browser.current_url}')
            html = browser.page_source.encode('utf-8')
            page, cards = dzen_parse_cards(html, url)
            if not cards:
                raise ValueError('Карточки рубрики не найдены')
            before = len(seen)
            stamp = datetime.now(timezone.utc).isoformat()
            for card in cards:
                seen.add(card['story_id'])
                if card['story_id'] not in catalog:
                    card.update(collected_at=stamp, listing_url=url, fetch_method='selenium', first_seen_run=run_id)
                    catalog[card['story_id']] = card
            growth = len(seen) - before
            if growth:
                (snapshots / f'step_{step:03d}.html').write_bytes(html)
            save_jsonl(path, catalog.values())
            metrics = browser.execute_script('return {top:document.scrollingElement.scrollTop,height:document.scrollingElement.scrollHeight,viewport:document.scrollingElement.clientHeight}')
            steps.append({'step': step, 'visible_cards': len(cards), 'seen_ids': len(seen), 'new_ids': growth, **metrics})
            print(f'Лента: {len(seen)} сюжетов сеанса, {len(catalog)} всего; прокрутка {step}', flush=True)
            idle = 0 if growth else idle + 1
            if len(seen) >= max_stories:
                reason = 'sample_limit'
                break
            if idle >= 3:
                reason = 'no_new_cards_observed'
                break
            if step >= max_scrolls:
                reason = 'scroll_limit'
                break
            browser.execute_script('window.scrollTo(0,document.scrollingElement.scrollHeight)')
            time.sleep(2)
    except Exception:
        reason = 'error'
        raise
    finally:
        log = {'run_id': run_id, 'listing_url': url, 'seen_this_run': len(seen),
               'catalog_total': len(catalog), 'stop_reason': reason,
               'archive_complete': False, 'steps': steps,
               'collected_at': datetime.now(timezone.utc).isoformat()}
        (output / f'dzen_listing_run_{run_id}.json').write_text(json.dumps(log, ensure_ascii=False, indent=2), encoding='utf-8')
        browser.quit()
    print('Причина остановки:', reason, flush=True)


collect_dzen_listing()


In [ ]:
# 2. Данные сюжетов только из Дзена; внешние статьи СМИ не загружаем.
# Функции разбора карточек определены в предыдущей ячейке.
def clean_url(url):
    p = urlsplit(url)
    if p.hostname == 'dzen.ru' and p.path == '/away':
        target = dict(parse_qsl(p.query)).get('to')
        if target:
            return clean_url(target)
    return urlunsplit((p.scheme, p.netloc, p.path,
                      urlencode([(k, v) for k, v in parse_qsl(p.query)
                                 if not k.startswith('utm_')]), ''))


def find_story(page, record):
    # Проверяем ID самой страницы, чтобы не взять текст из рекомендаций.
    canonical = page.select_one('link[rel="canonical"]')
    target = urlsplit(canonical.get('href', '')) if canonical else None
    requested_path = f"/news/story/{record['story_id']}"
    if target and target.hostname == 'dzen.ru' and target.path.rstrip('/') == requested_path:
        section = page.select_one('[data-testid="story-container"]')
        if section is not None:
            return section
    raise ValueError('ID основного документа не совпадает с ID нужного сюжета')


def parse_story(html, record):
    page = BeautifulSoup(html, 'html.parser')
    section = find_story(page, record)
    body = section.select_one('[data-testid="full-text-story"] [data-testid="article-body"]')
    primary = section.select_one('h1 a[href]')
    publisher = section.select_one('[data-testid="link-smi-name"]')
    sources = []
    seen = set()
    # Сначала основная публикация, затем остальные видимые источники.
    links = ([primary] if primary else []) + list(section.select(
        'a[data-testid="story-source-link"][href],a[data-testid="source-link"][href]'))
    for link in links:
        url = clean_url(urljoin(record['story_url'], link['href']))
        if urlsplit(url).scheme not in ('http', 'https') or url in seen:
            continue
        seen.add(url)
        label = publisher.get('aria-label', '').removeprefix('Источник: ') if link is primary and publisher else soup_text(link)
        sources.append({'url': url, 'publisher': label})
    snippets = []
    for item in section.select('[data-testid="summarization-item"]'):
        copied = BeautifulSoup(str(item), 'html.parser')
        citation = copied.select_one('[data-testid="source-link"]')
        label = soup_text(citation)
        citation_url = clean_url(urljoin(record['story_url'], citation['href'])) if citation and citation.get('href') else None
        if citation:
            citation.decompose()
        snippets.append({'text': soup_text(copied), 'publisher': label, 'url': citation_url})
    all_sources = section.select_one('[data-testid="show-all-sources-iniline"][href]')
    date = page.select_one('meta[property="article:published_time"]')
    return {'resolved_story_title': soup_text(section.select_one('h1')), 'text': soup_text(body), 'text_origin': 'dzen_article_body' if body else None,
            'selected_publisher': publisher.get('aria-label', '').removeprefix('Источник: ') if publisher else None,
            'article_url': clean_url(urljoin(record['story_url'], primary['href'])) if primary else None,
            'visible_sources': sources, 'story_snippets': snippets,
            'all_sources_url': urljoin(record['story_url'], all_sources['href']) if all_sources else None,
            'published_at': date.get('content') if date else None}


# Один браузер используется повторно, если requests не справился.
dzen_browser = None


def load_dzen_story(entry, raw, session):
    global dzen_browser
    url = entry['story_url']
    file_id = hashlib.sha256(url.encode()).hexdigest()[:20]
    path = raw / f'story_{file_id}.html'
    errors = []

    # 1. Сначала пробуем сохранённый HTML.
    if path.exists():
        try:
            return parse_story(path.read_bytes(), entry), 'cache', errors
        except ValueError as error:
            errors.append(f'cache: {error}')

    # 2. Затем обычный HTTP-запрос.
    time.sleep(REQUEST_DELAY)
    try:
        response = session.get(url, timeout=(4, 8))
        response.raise_for_status()
        story = parse_story(response.content, entry)
        path.write_bytes(response.content)
        return story, 'requests', errors
    except (requests.RequestException, ValueError) as error:
        errors.append(f'requests: {error}')

    # 3. Если не получилось, открываем страницу в браузере.
    if dzen_browser is None:
        options = webdriver.ChromeOptions()
        options.page_load_strategy = 'eager'
        if not DZEN_VISIBLE_BROWSER:
            options.add_argument('--headless=new')
        dzen_browser = webdriver.Chrome(options=options)
        dzen_browser.set_page_load_timeout(20)
    time.sleep(REQUEST_DELAY)
    try:
        try:
            dzen_browser.get(url)
        except TimeoutException:
            pass

        def story_ready(driver):
            try:
                parse_story(driver.page_source, entry)
                return True
            except ValueError:
                return False

        WebDriverWait(dzen_browser, 10).until(story_ready)
        html = dzen_browser.page_source
        story = parse_story(html, entry)
        path.write_text(html, encoding='utf-8')
        return story, 'selenium', errors
    except (WebDriverException, ValueError) as error:
        try:
            path.with_suffix('.error.html').write_text(dzen_browser.page_source, encoding='utf-8')
        except WebDriverException:
            pass
        errors.append(f'selenium: {type(error).__name__}: {error}')
        raise ValueError('; '.join(errors)) from error


def collect_dzen_stories():
    global dzen_browser
    raw = RAW_DIR / 'dzen/stories'
    raw.mkdir(parents=True, exist_ok=True)
    catalog = read_jsonl(WORK_DIR / 'dzen_stories_listing.jsonl')
    output = FINAL_DIR / 'dzen.jsonl'
    stories = {r['story_id']: r for r in read_jsonl(output)}
    processed_count = 0
    session = requests.Session()
    try:
        for index, entry in enumerate(catalog, 1):
            previous = stories.get(entry['story_id'], {})
            if not DZEN_REPARSE and previous.get('fetch_status') == 'ok' and previous.get('collection_scope') == 'dzen_only':
                continue
            if DZEN_PROCESS_LIMIT is not None and processed_count >= DZEN_PROCESS_LIMIT:
                break
            processed_count += 1
            started = time.monotonic()
            record = dict(entry)
            errors = []
            try:
                # Сохраняем совместимость с HTML, скачанным прежними версиями ячейки.
                story = None
                for prefix in ['story', 'selenium', 'requests']:
                    cached = RAW_DIR / f"dzen/{prefix}_{entry['story_id']}.html"
                    if cached.exists():
                        try:
                            story = parse_story(cached.read_bytes(), entry)
                            break
                        except ValueError:
                            pass
                method = 'cache'
                if story is None:
                    story, method, errors = load_dzen_story(entry, raw, session)
                record.update(story)
                if story['text']:
                    text_status = 'full_text_available'
                elif story['story_snippets']:
                    text_status = 'summary_only'
                else:
                    text_status = 'metadata_only'
                record.update(
                    story_fetch_method=method, fetch_status='ok', fetch_error=None,
                    collection_scope='dzen_only', text_status=text_status,
                    collection_note=None if story['text'] else
                        'Полный текст в Дзене недоступен; внешние статьи СМИ не учитываются.'
                )
            except (ValueError, requests.RequestException, WebDriverException) as error:
                record.update(fetch_status='error', fetch_error=str(error), collection_scope='dzen_only')
            record.update(enrichment_errors=errors, full_collected_at=datetime.now(timezone.utc).isoformat(),
                          duration_seconds=round(time.monotonic() - started, 2))
            stories[entry['story_id']] = record
            save_jsonl(output, stories.values())
            print(f"{index}/{len(catalog)}: {record['fetch_status']} — {entry['title']}")
            if record.get('fetch_error'):
                print('Причина:', record['fetch_error'][:250])
    finally:
        session.close()
        if dzen_browser is not None:
            try:
                dzen_browser.quit()
            except WebDriverException:
                pass
            dzen_browser = None
    print('Итог:', dict(Counter(r.get('fetch_status') for r in stories.values())))


collect_dzen_stories()


<a id="code-source-4"></a>
### 4PDA

[К текстовой части](#solution-text)


In [ ]:
# 4PDA: список тем → первая и 10 последних страниц → сообщения → JSONL.
# Настройки и общие функции находятся в начале ноутбука.


raw_folder = RAW_DIR / '4pda'
results_folder = WORK_DIR
raw_folder.mkdir(parents=True, exist_ok=True)
results_folder.mkdir(parents=True, exist_ok=True)


# 1. Ссылки и выбор страниц темы.
def forum_url(**parameters):
    return FORUM_URL + '?' + urlencode(parameters)


def page_offsets(last_offset):
    # Первая страница плюс N последних, включая последнюю неполную.
    # set исключает повтор первой страницы в коротких темах.
    if last_offset < 0 or last_offset % POSTS_PER_PAGE:
        raise ValueError('Смещение не соответствует шагу пагинации; проверь POSTS_PER_PAGE.')
    first_recent = max(0, last_offset - (LAST_PAGES - 1) * POSTS_PER_PAGE)
    recent_pages = list(range(first_recent, last_offset + 1, POSTS_PER_PAGE))
    return sorted(set([0] + recent_pages))


# 2. Из списка тем достаём название, автора, счётчики и последнюю страницу.
def parse_topics(html):
    soup = BeautifulSoup(html, 'html.parser')
    topics = {}
    for anchor in soup.select('a[id^="tid-link-"][href]'):
        query = parse_qs(urlsplit(urljoin(FORUM_URL, anchor['href'])).query)
        ids = query.get('showtopic', [])
        if len(ids) != 1 or not ids[0].isdigit():
            continue
        tid = ids[0]
        row = anchor.find_parent('tr')
        cells = row.find_all('td', recursive=False) if row else []
        author = cells[4].select_one('a[href*="showuser="]') if len(cells) >= 7 else None
        offsets = [0]
        for link in row.select('a[href]') if row else []:
            q = parse_qs(urlsplit(urljoin(FORUM_URL, link['href'])).query)
            values = q.get('st', [])
            if q.get('showtopic') == [tid] and len(values) == 1 and values[0].isdigit():
                offsets.append(int(values[0]))
        topics[tid] = {
            'source': '4pda', 'forum_id': FORUM_ID, 'topic_id': tid,
            'title': get_text(anchor), 'url': forum_url(showtopic=tid),
            'description': get_text(row.select_one(f'[id="tid-desc-{tid}"]')) if row else None,
            'created_date_raw': anchor.get('title', '').removeprefix('Тема создана ') or None,
            'starter': get_text(author),
            'starter_url': urljoin(FORUM_URL, author['href']) if author else None,
            'replies_count': get_number(get_text(cells[3])) if len(cells) >= 7 else None,
            'views_count': get_number(get_text(cells[5])) if len(cells) >= 7 else None,
            'last_page_offset_seen': max(offsets),
        }
    if not topics:
        raise ValueError('Темы раздела не найдены: возможно, вместо форума получена проверка сайта.')
    return list(topics.values())


# 3. Из страницы темы достаём отдельные сообщения.
def get_post_id(block):
    # Сначала постоянный якорь/ссылка сообщения, затем data-post.
    for anchor in block.select('a[name],a[id]'):
        marker = anchor.get('name') or anchor.get('id') or ''
        match = re.fullmatch(r'entry(\d+)', marker)
        if match:
            return match.group(1), 'entry_anchor'
    for anchor in block.select('a[href]'):
        target = urlsplit(urljoin(FORUM_URL, anchor['href']))
        match = re.fullmatch(r'entry(\d+)', target.fragment)
        query = parse_qs(target.query)
        values = query.get('p', [])
        if match:
            return match.group(1), 'entry_link'
        if len(values) == 1 and values[0].isdigit() and query.get('view') == ['findpost']:
            return values[0], 'permalink'
    raw = block.get('data-post', '')
    if isinstance(raw, str) and raw.isdecimal():
        return raw, 'data-post'
    try:
        data = json.loads(raw)
    except (ValueError, TypeError):
        data = None
    if isinstance(data, dict):
        for key in ['pid', 'post_id', 'id']:
            if str(data.get(key, '')).isdecimal():
                return str(data[key]), f'data-post.{key}'
    return None, None


def parse_posts(html, topic, page_url):
    soup = BeautifulSoup(html, 'html.parser')
    # Классическая разметка IPB: отдельная таблица на сообщение.
    blocks = soup.select('table.ipbtable[data-post]')
    if not blocks:
        blocks = []
        seen = set()
        for body in soup.select('.postcolor'):
            block = body.find_parent('table', class_='ipbtable')
            if block is not None and id(block) not in seen:
                blocks.append(block)
                seen.add(id(block))
    if not blocks:
        raise ValueError('Блоки сообщений не найдены. HTML сохранён; нельзя считать проверку сайта сообщениями.')
    records = []
    for block in blocks:
        body = block.select_one('.postcolor')
        if body is None:
            continue
        # Не ищем автора и дату внутри текста/цитат самого сообщения.
        header = BeautifulSoup(str(block), 'html.parser')
        for element in header.select('.postcolor, .signature'):
            element.decompose()
        author = None
        for link in header.select('a[href*="showuser="]'):
            if get_text(link):
                author = link
                break
        date_pattern = r'(?:\d{1,2}\.\d{1,2}\.\d{2,4}|Сегодня|Вчера),?\s+\d{1,2}:\d{2}'
        date_match = re.search(date_pattern, get_text(header) or '', re.I)
        date = date_match.group(0) if date_match else None
        pid, pid_source = get_post_id(header.select_one("table.ipbtable") or header)
        quotes = [get_text(q) for q in body.select('.quotemain')]
        cleaned = BeautifulSoup(str(body), 'html.parser')
        for element in cleaned.select('script,style,noscript,form,button,.signature'):
            element.decompose()
        text = get_text(cleaned) or ''
        images = []
        for image in body.select('img'):
            src = image.get('data-src') or image.get('src')
            if src:
                images.append(urljoin(page_url, src))
        attachments = []
        for link in body.select('a[href]'):
            if 'act=attach' in link['href']:
                attachments.append(urljoin(page_url, link['href']))
        # Если постоянный ID отсутствует, не выдумываем его: отдельный ключ дедупликации.
        identity = [topic['topic_id'], get_text(author), date, text]
        fallback = hashlib.sha256(json.dumps(identity, ensure_ascii=False).encode()).hexdigest()
        post_key = f"{topic['topic_id']}:{pid}" if pid else f"hash:{fallback}"
        records.append({
            'source': '4pda', 'forum_id': FORUM_ID,
            'topic_id': topic['topic_id'], 'topic_title': topic['title'],
            'post_id': pid, 'post_id_source': pid_source, 'post_key': post_key,
            'url': forum_url(showtopic=topic['topic_id'], view='findpost', p=pid) if pid else page_url,
            'page_url': page_url, 'author': get_text(author),
            'author_url': urljoin(page_url, author['href']) if author else None,
            'published_date_raw': date,
            'text': text, 'quoted_texts': quotes, 'quotes_included_in_text': bool(quotes),
            'image_urls': list(dict.fromkeys(images)),
            'attachment_urls': list(dict.fromkeys(attachments)),
            'collected_at': datetime.now(timezone.utc).isoformat(),
        })
    if not records:
        raise ValueError('Таблицы найдены, но тексты сообщений не распознаны; нужна проверка HTML.')
    return records


# 4. Загружаем прежний результат: после остановки продолжим с нужного места.
posts_file = FINAL_DIR / '4pda.jsonl'
posts = {r['post_key']: r for r in read_jsonl(posts_file)}
progress_file = results_folder / '4pda_collection.json'
progress = {'forum_id': FORUM_ID, 'scope': f'First {TOPICS_LIMIT} topics; first page and last {LAST_PAGES} pages per topic, without duplicates',
             'last_pages_per_topic': LAST_PAGES, 'posts_per_page': POSTS_PER_PAGE,
             'archive_complete': False, 'pages': {}, 'started_at': datetime.now(timezone.utc).isoformat()}
if progress_file.exists():
    previous = json.loads(progress_file.read_text(encoding='utf-8'))
    progress['pages'] = previous.get('pages', {})

browser = None
manual_check_used = False


def save_progress():
    save_jsonl(posts_file, posts.values())
    progress['posts_count'] = len(posts)
    progress['missing_author'] = sum(not p.get('author') for p in posts.values())
    progress['missing_date'] = sum(not p.get('published_date_raw') for p in posts.values())
    progress['missing_post_id'] = sum(not p.get('post_id') for p in posts.values())
    temporary = progress_file.with_suffix('.json.tmp')
    temporary.write_text(json.dumps(progress, ensure_ascii=False, indent=2), encoding='utf-8')
    temporary.replace(progress_file)


# 5. Открываем страницу или берём её сохранённый HTML.
# CSS-селекторы ниже — названия элементов сайта, по ним BeautifulSoup/Selenium находят данные.
def load_page(url, path, selector, use_cache=True):
    global manual_check_used
    if use_cache and path.exists():
        html = path.read_text(encoding='utf-8')
        if BeautifulSoup(html, 'html.parser').select_one(selector):
            return html
    try:
        browser.get(url)
    except TimeoutException:
        pass  # Иногда загрузка ресурсов зависает, хотя сообщения уже появились.

    def page_ready(driver):
        expected = parse_qs(urlsplit(url).query)
        actual = parse_qs(urlsplit(driver.current_url).query)
        for key in ['showforum', 'showtopic', 'st']:
            if key in expected and actual.get(key, ['0']) != expected[key]:
                return False
        return driver.find_elements('css selector', selector)

    try:
        WebDriverWait(browser, 20).until(page_ready)
    except TimeoutException:
        if manual_check_used:
            path.write_text(browser.page_source, encoding='utf-8')
            raise RuntimeError('Страница не загрузилась или доступ ограничен. HTML и прогресс сохранены.')
        manual_check_used = True
        print('В Chrome пока нет нужных данных:', browser.title)
        input('Пройди проверку сайта в этом окне Chrome, затем нажми Enter здесь: ')
        try:
            WebDriverWait(browser, 45).until(page_ready)
        except TimeoutException:
            path.write_text(browser.page_source, encoding='utf-8')
            raise
    html = browser.page_source
    path.write_text(html, encoding='utf-8')
    return html


# 6. Основной сбор: список тем → страницы → сообщения → сохранение.
options = webdriver.ChromeOptions()
options.page_load_strategy = 'eager'


try:
    browser = webdriver.Chrome(options=options)
    browser.set_page_load_timeout(30)
    listing_url = forum_url(showforum=FORUM_ID)
    listing_html = load_page(listing_url, raw_folder / 'forum_320.html', 'a[id^="tid-link-"]', use_cache=False)
    topics = parse_topics(listing_html)[:TOPICS_LIMIT]
    save_jsonl(results_folder / '4pda_topics.jsonl', topics)
    print('Тем в выборке:', len(topics))
    for topic_number, topic in enumerate(topics, 1):
        offsets = page_offsets(topic['last_page_offset_seen'])
        for offset in offsets:
            page_url = forum_url(showtopic=topic['topic_id'], st=offset)
            cached = raw_folder / f"topic_{topic['topic_id']}_st_{offset}.html"
            key = f"{topic['topic_id']}:{offset}"
            if progress['pages'].get(key, {}).get('status') == 'ok':
                continue
            try:
                html = load_page(page_url, cached, '.postcolor')
                parsed_posts = parse_posts(html, topic, page_url)
                for post in parsed_posts:
                    posts[post['post_key']] = post
                progress['pages'][key] = {'url': page_url, 'status': 'ok', 'posts': len(parsed_posts)}
                save_progress()
                print(f"Тема {topic_number}/{len(topics)}, st={offset}: {len(parsed_posts)} сообщений; уникальных всего {len(posts)}")
                time.sleep(1)
            except Exception as error:
                progress['pages'][key] = {'url': page_url, 'status': 'error', 'error': f'{type(error).__name__}: {error}'}
                save_progress()
                print('Остановка:', error)
                raise  # Не продолжаем десятки неудачных запросов.
finally:
    save_progress()
    if browser is not None:
        browser.quit()

print('Сбор выборки завершён:', len(posts), 'уникальных сообщений.')
print('Нет автора:', progress['missing_author'], '; нет даты:', progress['missing_date'], '; нет ID:', progress['missing_post_id'])
print('Результат:', posts_file)


<a id="code-source-5"></a>
### Отзовик

[К текстовой части](#solution-text)


In [52]:
# 1. Разбираем карточки каталога: название, ссылка и число отзывов.
def parse_otzovik_catalog(html, listing_url):
    page = BeautifulSoup(html, 'html.parser')
    cards = page.select('.product-list .item[data-pid]')
    if not cards:
        raise ValueError('Карточки каталога не найдены: проверь HTML или доступ к сайту.')
    selected = []
    seen_urls = set()
    for card in cards:
        link = card.select_one('a.product-name[href]')
        count_text = get_text(card.select_one('.reviews-counter')) or ''
        match = re.search(r'([\d\s]+)\s*отзыв', count_text, re.I)
        if link is None or match is None:
            raise ValueError('У карточки не распознаны название или число отзывов.')
        object_url = urljoin(OTZOVIK_URL, link['href'])
        target = urlsplit(object_url)
        if target.hostname != 'otzovik.com' or not re.fullmatch(r'/reviews/[^/]+/', target.path):
            raise ValueError('Неожиданная ссылка объекта: ' + object_url)
        seen_urls.add(object_url)
        count = get_number(match.group(1))
        if count < OTZOVIK_MIN_REVIEWS:
            continue  # Такие объекты вообще не включаем в JSONL.
        selected.append({
            'source': 'otzovik', 'object_id': card.get('data-pid'),
            'object_url': object_url, 'name': get_text(link), 'reviews_count': count,
            'category_url': OTZOVIK_URL, 'listing_url': listing_url,
            'collected_at': datetime.now(timezone.utc).isoformat(),
        })
    next_pages = set()
    for link in page.select('a.pager-item[href], a.next[href]'):
        target = urlsplit(urljoin(listing_url, link['href']))
        match = re.fullmatch(r'/money_systems/banks/(\d+)/', target.path)
        if target.hostname == 'otzovik.com' and match:
            next_pages.add(int(match.group(1)))
    return selected, seen_urls, sorted(next_pages)


# 2. Загрузка HTML. Браузер создаём только после неудачного requests.
def load_otzovik_page(url, path, selector):
    global otzovik_browser, otzovik_use_requests, otzovik_manual_used
    if path.exists():
        html = path.read_text(encoding='utf-8')
        if BeautifulSoup(html, 'html.parser').select_one(selector):
            return html
    if otzovik_use_requests:
        time.sleep(REQUEST_DELAY)
        try:
            response = otzovik_session.get(url, timeout=(5, 20))
            response.raise_for_status()
            actual, expected = urlsplit(response.url), urlsplit(url)
            if actual.hostname != expected.hostname or actual.path.rstrip('/') != expected.path.rstrip('/'):
                raise ValueError('Сайт вернул другую страницу: ' + response.url)
            html = response.text
            if not BeautifulSoup(html, 'html.parser').select_one(selector):
                raise ValueError('В ответе нет нужных элементов страницы')
            path.write_text(html, encoding='utf-8')
            return html
        except (requests.RequestException, ValueError) as error:
            print('Переходим к браузеру:', error)
            otzovik_use_requests = False
    if otzovik_browser is None:
        options = webdriver.ChromeOptions()
        options.page_load_strategy = 'eager'
        otzovik_browser = webdriver.Chrome(options=options)
        otzovik_browser.set_page_load_timeout(30)
    time.sleep(REQUEST_DELAY)
    try:
        otzovik_browser.get(url)
    except TimeoutException:
        pass

    def page_ready(driver):
        actual, expected = urlsplit(driver.current_url), urlsplit(url)
        same_page = actual.hostname == expected.hostname and actual.path.rstrip('/') == expected.path.rstrip('/')
        return same_page and driver.find_elements(By.CSS_SELECTOR, selector)

    try:
        WebDriverWait(otzovik_browser, 20).until(page_ready)
    except TimeoutException:
        # Сохраняем страницу до любых ручных действий.
        path.write_text(otzovik_browser.page_source, encoding='utf-8')
        loaded_page = BeautifulSoup(otzovik_browser.page_source, 'html.parser')
        if loaded_page.select_one('.review-contents[data-rid]'):
            raise ValueError(f'Отзыв открыт, но элемент {selector} не найден. HTML: {path}')
        if otzovik_manual_used:
            path.write_text(otzovik_browser.page_source, encoding='utf-8')
            raise RuntimeError('Доступ снова ограничен или разметка изменилась. Сбор остановлен.')
        otzovik_manual_used = True
        input('Пройди проверку в открытом Chrome и нажми Enter здесь: ')
        try:
            WebDriverWait(otzovik_browser, 45).until(page_ready)
        except TimeoutException as error:
            path.write_text(otzovik_browser.page_source, encoding='utf-8')
            raise RuntimeError(
                f'После ручной проверки нет нужного элемента: {selector}. '
                f'Страница: {otzovik_browser.title}. HTML: {path}'
            ) from error
    html = otzovik_browser.page_source
    path.write_text(html, encoding='utf-8')
    return html


In [53]:
# 3. Обходим каталог, фильтруем объекты и сохраняем после каждой страницы.
otzovik_raw = RAW_DIR / 'otzovik'
objects_path = WORK_DIR / 'otzovik_objects.jsonl'
progress_path = WORK_DIR / 'otzovik_catalog_progress.json'
# Старый каталог без reviews_count перепроверим по HTML; неподходящие записи не переносим.
objects = {}
for item in read_jsonl(objects_path):
    if (item.get('reviews_count') or 0) >= OTZOVIK_MIN_REVIEWS:
        objects[item['object_url']] = item
pending = deque([1])
scheduled = {1}
seen_objects = set()
page_log = []
catalog_complete = False
otzovik_browser = None
otzovik_use_requests = True
otzovik_manual_used = False
otzovik_session = requests.Session()
try:
    while pending:
        if len(page_log) >= OTZOVIK_CATALOG_LIMIT:
            raise RuntimeError('Достигнут лимит страниц; каталог собран не полностью.')
        number = pending.popleft()
        listing_url = OTZOVIK_URL if number == 1 else f'{OTZOVIK_URL}{number}/'
        path = otzovik_raw / ('banks.html' if number == 1 else f'banks_{number}.html')
        html = load_otzovik_page(listing_url, path, '.product-list .item[data-pid]')
        selected, seen, next_pages = parse_otzovik_catalog(html, listing_url)
        seen_objects.update(seen)
        # Если число отзывов у объекта изменилось, учитываем текущий счётчик.
        for object_url in seen:
            objects.pop(object_url, None)
        for item in selected:
            objects[item['object_url']] = item
        for next_number in next_pages:
            if next_number not in scheduled:
                scheduled.add(next_number)
                pending.append(next_number)
        page_log.append({'url': listing_url, 'objects': len(seen), 'selected': len(selected)})
        save_jsonl(objects_path, objects.values())
        print(f'Страница {number}: объектов {len(seen)}, подходят {len(selected)}, сохранено всего {len(objects)}')
    catalog_complete = True
    # Заранее сохраняем страницу одного подходящего объекта для проверки разметки отзывов.
    if objects:
        first = next(iter(objects.values()))
        load_otzovik_page(first['object_url'], otzovik_raw / 'sample_object.html', 'a[href*="/review_"]')
        print('HTML объекта для следующего этапа сохранён:', first['name'])
finally:
    save_jsonl(objects_path, objects.values())
    progress = {
        'catalog_complete': catalog_complete, 'pages': page_log,
        'objects_seen_this_run': len(seen_objects), 'selected_objects': len(objects),
        'min_reviews': OTZOVIK_MIN_REVIEWS, 'reviews_limit_per_object': OTZOVIK_REVIEWS_LIMIT,
        'collected_at': datetime.now(timezone.utc).isoformat(),
    }
    temporary = progress_path.with_suffix('.json.tmp')
    temporary.write_text(json.dumps(progress, ensure_ascii=False, indent=2), encoding='utf-8')
    temporary.replace(progress_path)
    otzovik_session.close()
    if otzovik_browser is not None:
        try:
            otzovik_browser.quit()
        except WebDriverException:
            pass  # Ошибка закрытия не должна скрывать причину остановки.
        otzovik_browser = None
print('Каталог завершён:', catalog_complete, '; объектов с 10+ отзывами:', len(objects))
print('Далее: до', OTZOVIK_REVIEWS_LIMIT, 'последних отзывов по каждому выбранному объекту.')


Страница 1: объектов 30, подходят 1, сохранено всего 395
Страница 2: объектов 30, подходят 2, сохранено всего 395
Страница 3: объектов 30, подходят 5, сохранено всего 395
Страница 4: объектов 30, подходят 4, сохранено всего 395
Страница 5: объектов 30, подходят 4, сохранено всего 395
Страница 6: объектов 30, подходят 6, сохранено всего 395
Страница 7: объектов 30, подходят 6, сохранено всего 395
Страница 50: объектов 3, подходят 1, сохранено всего 395
Страница 8: объектов 30, подходят 11, сохранено всего 395
Страница 9: объектов 30, подходят 5, сохранено всего 395
Страница 10: объектов 30, подходят 4, сохранено всего 395
Страница 44: объектов 30, подходят 12, сохранено всего 395
Страница 45: объектов 30, подходят 14, сохранено всего 395
Страница 46: объектов 30, подходят 9, сохранено всего 395
Страница 47: объектов 30, подходят 11, сохранено всего 395
Страница 48: объектов 30, подходят 6, сохранено всего 395
Страница 49: объектов 30, подходят 13, сохранено всего 395
Страница 11: объект

In [54]:
# 1. Ссылки на отзывы из списка «Сначала новые».
def parse_otzovik_review_list(html, item, listing_url, require_newest=True):
    page = BeautifulSoup(html, 'html.parser')
    sort = page.select_one('input[name="order"][value="date_desc"][checked]')
    current = page.select_one('.product-sort-popup a.current[href]')
    current_order = parse_qs(urlsplit(current['href']).query).get('order') if current else None
    if require_newest and sort is None and current_order != ['date_desc']:
        raise ValueError('Не подтверждена сортировка «Сначала новые». HTML сохранён для проверки.')
    identity = page.select_one('#reviews-sort-tools[data-prod-root]')
    if identity and urlsplit(identity['data-prod-root']).path.rstrip('/') != urlsplit(item['object_url']).path.rstrip('/'):
        raise ValueError('Страница относится к другому объекту.')
    records = []
    for card in page.select('.review-list-2 .item[itemprop="review"]'):
        link = card.select_one('a.review-title[href]')
        if link is None:
            continue
        url = urljoin(listing_url, link['href'])
        match = re.fullmatch(r'/review_(\d+)\.html', urlsplit(url).path)
        if urlsplit(url).hostname != 'otzovik.com' or match is None:
            continue
        author = card.select_one('a.user-login[href]')
        date = card.select_one('[itemprop="datePublished"]')
        rating = card.select_one('[itemprop="ratingValue"]')
        records.append({
            'source': 'otzovik', 'object_id': item['object_id'],
            'object_name': item['name'], 'object_url': item['object_url'],
            'review_id': match.group(1), 'url': url, 'title': get_text(link),
            'author': get_text(author),
            'author_url': urljoin(url, author['href']) if author else None,
            'published_at': date.get('content') if date else None,
            'published_date_display': get_text(date),
            'rating': get_number(rating.get('content')) if rating else None,
            'teaser': get_text(card.select_one('.review-teaser')),
            'advantages': get_text(card.select_one('.review-plus')),
            'disadvantages': get_text(card.select_one('.review-minus')),
            'comments_count': get_number(get_text(card.select_one('[itemprop="commentCount"]'))),
            'helpful_count': get_number(get_text(card.select_one('.review-yes span'))),
            'listing_url': listing_url,
        })
    if not records:
        raise ValueError('Карточки отзывов не найдены. Проверяем сохранённый HTML.')
    # Берём только ссылки пагинации того же объекта; порядок по дате сохраняем.
    root_path = urlsplit(item['object_url']).path.rstrip('/')
    next_pages = set()
    for link in page.select('a[href]'):
        target = urlsplit(urljoin(listing_url, link['href']))
        match = re.fullmatch(re.escape(root_path) + r'/(\d+)/?', target.path)
        if target.hostname == 'otzovik.com' and match:
            number = int(match.group(1))
            next_pages.add((number, f"https://otzovik.com{target.path}?order=date_desc"))
    return records, sorted(next_pages)


# 2. Полный текст отдельного отзыва. Отрывок из списка сюда не подставляем.
def parse_otzovik_full_review(html, url):
    page = BeautifulSoup(html, 'html.parser')
    canonical = page.select_one('link[rel="canonical"][href]')
    if canonical and urlsplit(canonical['href']).path != urlsplit(url).path:
        raise ValueError('Открыт другой отзыв.')
    body = page.select_one('.review-contents .review-body, [itemprop="reviewBody"]')
    if body is None:
        raise ValueError('Полный текст .review-body не найден. Проверь сохранённый HTML.')
    block = body.find_parent(class_='review-contents')
    if block is None:
        block = body.find_parent(attrs={'itemtype': re.compile(r'/Review$')})
    if block is None:
        block = page
    identity = block.select_one('link[itemprop="url"][href]')
    if identity and urlsplit(identity['href']).path != urlsplit(url).path:
        raise ValueError('Блок текста относится к другому отзыву.')
    expected_id = re.fullmatch(r'/review_(\d+)\.html', urlsplit(url).path)
    if block.get('data-rid') and (expected_id is None or block['data-rid'] != expected_id.group(1)):
        raise ValueError('ID блока не совпадает с ID нужного отзыва.')
    images = []
    for image in body.select('img'):
        src = image.get('data-src') or image.get('data-original') or image.get('src')
        if src:
            images.append(urljoin(url, src))
    cleaned = BeautifulSoup(str(body), 'html.parser')
    for tag in cleaned.select('script,style,noscript,svg,button'):
        tag.decompose()
    text = cleaned.get_text('\n', strip=True)
    if not text:
        raise ValueError('Полный текст пустой.')
    author = block.select_one('a.user-login[href]')
    if author is None:
        author = block.select_one('[itemprop="author"] a[href*="/profile/"]')
    date = block.select_one('[itemprop="datePublished"]')
    rating = block.select_one('[itemprop="ratingValue"]')
    return {
        'text': text, 'full_title': get_text(block.select_one('h1')),
        'full_author': get_text(author),
        'full_author_url': urljoin(url, author['href']) if author else None,
        'full_published_at': date.get('content') if date else None,
        'full_rating': get_number(rating.get('content')) if rating else None,
        'full_advantages': get_text(block.select_one('.review-plus')),
        'full_disadvantages': get_text(block.select_one('.review-minus')),
        'image_urls': list(dict.fromkeys(images)),
    }


In [56]:
# 3. До 30 последних отзывов каждого объекта. Каталог повторно не обходим.
objects = [r for r in read_jsonl(WORK_DIR / 'otzovik_objects.jsonl')
           if r['reviews_count'] >= OTZOVIK_MIN_REVIEWS]
reviews_file = FINAL_DIR / 'otzovik.jsonl'
review_progress_file = WORK_DIR / 'otzovik_reviews_progress.json'
reviews = {r['url']: r for r in read_jsonl(reviews_file)}
object_progress = {}
if review_progress_file.exists():
    object_progress = json.loads(review_progress_file.read_text(encoding='utf-8')).get('objects', {})
otzovik_raw = RAW_DIR / 'otzovik'
(otzovik_raw / 'reviews').mkdir(parents=True, exist_ok=True)
otzovik_browser = None
otzovik_use_requests = True
otzovik_manual_used = False
otzovik_session = requests.Session()


def save_otzovik_reviews():
    save_jsonl(reviews_file, reviews.values())
    report = {
        'min_reviews': OTZOVIK_MIN_REVIEWS, 'reviews_limit_per_object': OTZOVIK_REVIEWS_LIMIT,
        'sort_order': 'date_desc', 'objects': object_progress,
        'ok_reviews': sum(r.get('fetch_status') == 'ok' for r in reviews.values()),
        'error_reviews': sum(r.get('fetch_status') == 'error' for r in reviews.values()),
        'collected_at': datetime.now(timezone.utc).isoformat(),
    }
    temporary = review_progress_file.with_suffix('.json.tmp')
    temporary.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
    temporary.replace(review_progress_file)


try:
    for number, item in enumerate(objects, 1):
        object_url = item['object_url']
        previous = object_progress.get(object_url, {})
        if previous.get('status') == 'ok' and previous.get('limit') == OTZOVIK_REVIEWS_LIMIT:
            print(f'Объект {number}/{len(objects)}: уже готов — {item["name"]}')
            continue
        object_progress[object_url] = {'status': 'running', 'limit': OTZOVIK_REVIEWS_LIMIT}
        try:
            # Сначала собираем ссылки в порядке публикации, максимум 30.
            selected = {}
            listing_pages = [(1, object_url + '?order=date_desc')]
            visited = set()
            expected_count = min(OTZOVIK_REVIEWS_LIMIT, item['reviews_count'])
            last_date = None
            while listing_pages and len(selected) < expected_count:
                page_number, listing_url = listing_pages.pop(0)
                if page_number in visited:
                    continue
                visited.add(page_number)
                cache = otzovik_raw / f"object_{item['object_id']}_date_desc_{page_number}.html"
                html = load_otzovik_page(listing_url, cache, '.review-list-2 a.review-title')
                cards, next_pages = parse_otzovik_review_list(html, item, listing_url)
                for record in cards:
                    if record['url'] in selected:
                        continue
                    # Проверяем порядок, чтобы не выдать полезные отзывы за последние.
                    if not record['published_at']:
                        raise ValueError('Не найдена дата публикации: порядок по дате проверить нельзя.')
                    published = datetime.fromisoformat(record['published_at'])
                    if last_date is not None and published > last_date:
                        raise ValueError('Отзывы идут не по убыванию даты. Нужна проверка сортировки.')
                    last_date = published
                    selected[record['url']] = record
                    if len(selected) >= expected_count:
                        break
                for next_number, next_url in next_pages:
                    if next_number > page_number and next_number not in visited:
                        listing_pages.append((next_number, next_url))
                listing_pages.sort()
            if len(selected) < expected_count:
                raise ValueError(f'Найдено {len(selected)} ссылок вместо {expected_count}; проверяем пагинацию.')

            # Затем скачиваем полные отзывы; успешные тексты повторно не запрашиваем.
            for review_number, record in enumerate(selected.values(), 1):
                old = reviews.get(record['url'], {})
                if old.get('fetch_status') == 'ok' and old.get('text'):
                    continue
                result = dict(record)
                try:
                    cache = otzovik_raw / 'reviews' / f"review_{record['review_id']}.html"
                    html = load_otzovik_page(record['url'], cache, '.review-contents .review-body, [itemprop="reviewBody"]')
                    result.update(parse_otzovik_full_review(html, record['url']))
                    result.update(fetch_status='ok', fetch_error=None)
                except Exception as error:
                    result.update(fetch_status='error', fetch_error=f'{type(error).__name__}: {error}',
                                  collected_at=datetime.now(timezone.utc).isoformat())
                    reviews[record['url']] = result
                    save_otzovik_reviews()
                    raise  # Не делаем сотни запросов, если сайт или разметка изменились.
                result['collected_at'] = datetime.now(timezone.utc).isoformat()
                reviews[record['url']] = result
                save_otzovik_reviews()
                print(f'Объект {number}/{len(objects)}, отзыв {review_number}/{len(selected)}: ok')
            object_progress[object_url] = {
                'status': 'ok', 'limit': OTZOVIK_REVIEWS_LIMIT,
                'selected_reviews': len(selected), 'listing_pages': len(visited),
            }
            save_otzovik_reviews()
            print(f'Готово: {item["name"]} — {len(selected)} отзывов')
        except Exception as error:
            object_progress[object_url] = {
                'status': 'error', 'limit': OTZOVIK_REVIEWS_LIMIT,
                'error': f'{type(error).__name__}: {error}',
            }
            save_otzovik_reviews()
            print('Остановка:', item['name'], '—', error)
            raise
finally:
    save_otzovik_reviews()
    otzovik_session.close()
    if otzovik_browser is not None:
        try:
            otzovik_browser.quit()
        except WebDriverException:
            pass  # Ошибка закрытия не должна скрывать причину остановки.
        otzovik_browser = None
print('Готово. Полные отзывы:', sum(r.get('fetch_status') == 'ok' for r in reviews.values()))
print('Файл:', reviews_file)


Объект 1/395: уже готов — Банк "Контур Банк" (Россия, Москва)
Объект 2/395: уже готов — МКК "Диджитал Мани" Бюджет
Объект 3/395: уже готов — Карта Азиатско-Тихоокеанского Банка Юнионпэй Дебетовая АТБ
Объект 4/395: уже готов — Служба поддержки Яндекс банк
Объект 5/395: уже готов — Общество с ограниченной ответственностью "Система"
Объект 6/395: уже готов — Ипотека в "Сбербанк" через "ДомКлик"
Объект 7/395: уже готов — Цифровые финансовые активы (ЦФА) Ozon Банк
Объект 8/395: уже готов — Расчетный счет для бизнеса Ozon банк
Объект 9/395: уже готов — Накопительный счет Ozon "Ежедневный доход"
Объект 10/395: уже готов — Банк "Газпромбанк" (Россия)
Объект 11/395: уже готов — Банк "Вайлдберриз" (Россия, Москва)
Объект 12/395: уже готов — Вклады на финуслугах Финуслуги
Объект 13/395: уже готов — НПФ Альфа
Объект 14/395: уже готов — Запрет на кредиты "Госуслуги"
Объект 15/395: уже готов — Накопительный счет МТС Банк "МТС счет"
Объект 16/395: уже готов — НПФ ВТБ (Россия, Москва)
Объект 17/395: у

KeyboardInterrupt: 

<a id="code-source-6"></a>
### DRIVE2

[К текстовой части](#solution-text)


In [ ]:
# 1. Разбор карточек, полного текста и загрузка страниц.
def parse_drive2_listing(html, url):
    page = BeautifulSoup(html, 'html.parser')
    if get_text(page.select_one('h1')) != 'Безопасность':
        raise ValueError('Не распознана подборка «Безопасность».')
    records = {}
    for card in page.select('.c-block-card'):
        link = card.select_one('.c-post-preview__title a[href]')
        if link is None:
            continue
        target = urlsplit(urljoin(url, link['href']))
        match = re.fullmatch(r'/(b|l)/(\d+)/', target.path)
        if target.hostname != 'www.drive2.ru' or match is None:
            continue
        post_url = 'https://www.drive2.ru' + target.path
        author = card.select_one('.c-block-card__meta .c-username[href]')
        likes = card.select_one('like-button[count]')
        preview = card.select_one('.c-post-preview__lead')
        if preview is not None:
            preview = BeautifulSoup(str(preview), 'html.parser')
            for button in preview.select('button'):
                button.decompose()
        records[post_url] = {
            'source': 'drive2', 'topic': 'Безопасность', 'topic_url': DRIVE2_URL,
            'post_id': match.group(2), 'post_type': match.group(1), 'url': post_url,
            'title': get_text(link), 'author': get_text(author),
            'author_url': urljoin(url, author['href']) if author else None,
            'listing_date_display': get_text(card.select_one('.c-author__date')),
            'teaser': get_text(preview), 'likes_count': get_number(likes.get('count')) if likes else None,
            'comments_count': get_number(get_text(card.select_one('.js-commentscounter'))),
            'listing_url': url, 'collected_at': datetime.now(timezone.utc).isoformat(),
        }
    if not records:
        raise ValueError('Карточки подборки не найдены.')
    link = page.select_one('.c-pager a[rel="next"][href]')
    next_url = urljoin(url, link['href']) if link else None
    if next_url:
        target = urlsplit(next_url)
        if target.hostname != 'www.drive2.ru' or target.path != urlsplit(DRIVE2_URL).path:
            raise ValueError('Пагинация ведёт за пределы нужной подборки.')
    return list(records.values()), next_url


def parse_drive2_post(html, entry):
    page = BeautifulSoup(html, 'html.parser')
    canonical = page.select_one('link[rel="canonical"][href]')
    if canonical is None or canonical['href'].rstrip('/') != entry['url'].rstrip('/'):
        raise ValueError('Canonical не совпадает с нужной публикацией.')
    post = page.select_one('.g-full-size-post')
    if post is None or post.get('id') != 'j' + entry['post_id']:
        raise ValueError('ID основного блока не совпадает с ID публикации.')
    body = post.select_one('[itemprop="articleBody"]')
    if body is None:
        # В обычных бортжурналах основной текст может не иметь articleBody.
        body = post.select_one('.c-post__body .c-link-decorated')
    if body is None:
        raise ValueError('Основной текст не найден.')
    date = post.select_one('meta[itemprop="datePublished"]')
    modified = post.select_one('meta[itemprop="dateModified"]')
    images = []
    for image in body.select('img[src]'):
        images.append(urljoin(entry['url'], image.get('data-src') or image['src']))
    links = []
    for anchor in body.select('a[href]'):
        target = urljoin(entry['url'], anchor['href'])
        if urlsplit(target).scheme in ('http', 'https'):
            links.append(target)
    videos = []
    for media in body.select('iframe[src],video[src],video source[src]'):
        videos.append(urljoin(entry['url'], media['src']))
    clean = BeautifulSoup(str(body), 'html.parser')
    for element in clean.select('script,style,noscript,svg,button,iframe'):
        element.decompose()
    full_text = clean.get_text('\n', strip=True)
    if not full_text:
        raise ValueError('Текст пустой.')
    # Метаданные читаем только из главного блока; ниже могут быть комментарии.
    likes = post.select_one('like-button[count]')
    return {
        'full_title': get_text(post.select_one('h1')), 'text': full_text,
        'published_at': date.get('content') if date else None,
        'published_date_display': get_text(post.select_one('.c-post-timestamp')),
        'modified_at': modified.get('content') if modified else None,
        'text_author': None,  # Аккаунт-публикатор и автор текста могут различаться.
        'image_urls': list(dict.fromkeys(images)), 'body_links': list(dict.fromkeys(links)),
        'video_urls': list(dict.fromkeys(videos)),
        'full_likes_count': get_number(likes.get('count')) if likes else None,
        'full_comments_count': get_number(get_text(post.select_one('.js-commentscounter'))),
    }


def load_drive2_page(browser, url, path, selector):
    if path.exists():
        html = path.read_text(encoding='utf-8')
        if BeautifulSoup(html, 'html.parser').select_one(selector):
            return html, 'cache'
    time.sleep(REQUEST_DELAY)
    try:
        browser.get(url)
    except TimeoutException:
        pass
    try:
        WebDriverWait(browser, 15).until(lambda driver: driver.find_elements('css selector', selector))
    finally:
        path.write_text(browser.page_source, encoding='utf-8')
    actual, expected = urlsplit(browser.current_url), urlsplit(url)
    if actual.hostname != expected.hostname or actual.path != expected.path or actual.query != expected.query:
        raise ValueError('Неожиданный адрес: ' + browser.current_url)
    return browser.page_source, 'selenium'




In [ ]:
# 2. Обход подборки и сбор публикаций. Готовые тексты пропускаются.
def collect_drive2():
    raw = PROJECT / 'data/raw/drive2'
    output = PROJECT / 'data/processed'
    raw.mkdir(parents=True, exist_ok=True)
    output.mkdir(parents=True, exist_ok=True)
    listing_file = output / 'drive2_listing.jsonl'
    posts_file = FINAL_DIR / 'drive2.jsonl'
    catalog = {r['url']: r for r in read_jsonl(listing_file)}
    posts = {r['url']: r for r in read_jsonl(posts_file)}
    pages = []
    next_url = DRIVE2_URL
    visited = set()
    browser = None
    report = {'topic_url': DRIVE2_URL, 'post_limit': DRIVE2_POST_LIMIT, 'page_limit': DRIVE2_PAGE_LIMIT,
              'archive_complete': False, 'pages': pages, 'stop_reason': 'error'}
    try:
        options = webdriver.ChromeOptions()
        options.page_load_strategy = 'eager'
        browser = webdriver.Chrome(options=options)
        browser.set_page_load_timeout(20)
        for page_number in range(1, DRIVE2_PAGE_LIMIT + 1):
            if not next_url:
                report['stop_reason'] = 'no_next_link'
                break
            if next_url in visited:
                raise ValueError('Повтор ссылки пагинации.')
            visited.add(next_url)
            url = next_url
            html, method = load_drive2_page(browser, url, raw / f'listing_{page_number:03}.html', '.c-post-preview__title a')
            cards, next_url = parse_drive2_listing(html, url)
            for card in cards:
                if len(catalog) >= DRIVE2_POST_LIMIT:
                    break
                catalog.setdefault(card['url'], card)
            pages.append({'url': url, 'cards': len(cards), 'method': method})
            save_jsonl(listing_file, catalog.values())
            print(f'Каталог: страница {page_number}, всего {len(catalog)} публикаций', flush=True)
            if len(catalog) >= DRIVE2_POST_LIMIT:
                report['stop_reason'] = 'sample_limit'
                break
        else:
            report['stop_reason'] = 'page_limit'
        for index, entry in enumerate(list(catalog.values())[:DRIVE2_POST_LIMIT], 1):
            old = posts.get(entry['url'], {})
            if old.get('fetch_status') == 'ok' and old.get('text'):
                continue
            record = dict(entry)
            try:
                html, method = load_drive2_page(browser, entry['url'], raw / f"post_{entry['post_id']}.html", '.g-full-size-post .c-post__body')
                record.update(parse_drive2_post(html, entry))
                record.update(fetch_status='ok', fetch_error=None, fetch_method=method)
            except (ValueError, WebDriverException) as error:
                record.update(fetch_status='error', fetch_error=f'{type(error).__name__}: {error}')
                posts[entry['url']] = record
                save_jsonl(posts_file, posts.values())
                raise
            record['full_collected_at'] = datetime.now(timezone.utc).isoformat()
            posts[entry['url']] = record
            save_jsonl(posts_file, posts.values())
            print(f"{index}/{len(catalog)}: ok, {len(record['text'])} символов — {entry['title']}", flush=True)
    finally:
        save_jsonl(posts_file, posts.values())
        report.update(posts_count=len(posts), ok_posts=sum(r.get('fetch_status') == 'ok' for r in posts.values()),
                      full_text_collection_complete=bool(catalog) and all(posts.get(url, {}).get('fetch_status') == 'ok' for url in catalog),
                      collected_at=datetime.now(timezone.utc).isoformat())
        (output / 'drive2_collection.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
        if browser:
            try:
                browser.quit()
            except WebDriverException:
                pass
    print(json.dumps(report, ensure_ascii=False), flush=True)

collect_drive2()


<a id="quality-check"></a>
### Проверка качества результатов

Достаточно запустить общие импорты и настройки, затем эту ячейку. Сбор сайтов повторять не нужно.


In [ ]:
# Проверка готовых JSONL. Ничего не скачивает и не изменяет исходные записи.
def quality_empty(value):
    return value is None or (isinstance(value, str) and not value.strip())


def quality_value(record, fields):
    # Поля полного текста имеют приоритет; при отсутствии берём поле карточки.
    for field in fields:
        value = record.get(field)
        if not quality_empty(value):
            return value
    return None


def quality_duplicates(values):
    counts = Counter(str(value) for value in values if not quality_empty(value))
    repeated = {value: count for value, count in counts.items() if count > 1}
    return {
        'extra_records': sum(count - 1 for count in repeated.values()),
        'repeated_values': repeated,
    }


quality_report = {
    'generated_at': datetime.now(timezone.utc).isoformat(),
    'missing_definition': 'Нет значения ни в одном из указанных полей: ключ отсутствует, null или пустая строка.',
    'duplicate_definition': 'Число записей сверх первого вхождения непустого ID или точного URL.',
    'sources': {},
}
quality_paths = sorted(FINAL_DIR.glob('*.jsonl'))
if not quality_paths:
    raise FileNotFoundError(f'В {FINAL_DIR} не найдены JSONL-файлы.')

for quality_path in quality_paths:
    records = []
    read_errors = []
    try:
        with quality_path.open(encoding='utf-8') as file:
            for line_number, line in enumerate(file, 1):
                if not line.strip():
                    continue
                try:
                    record = json.loads(line)
                    if not isinstance(record, dict):
                        raise ValueError('Строка JSONL должна содержать объект.')
                    records.append(record)
                except (ValueError, TypeError) as error:
                    read_errors.append({'line': line_number, 'reason': str(error)})
    except (OSError, UnicodeError) as error:
        read_errors.append({'line': None, 'reason': str(error)})

    fields = {
        'id': ['review_id', 'post_id', 'story_id'],
        'url': ['url', 'story_url'],
        'author': ['full_author', 'author', 'text_author'],
        'date': ['published_at', 'full_published_at', 'published_datetime_raw',
                 'published_date', 'published_date_raw'],
        'text': ['text'],
    }
    # У этих источников сохраняется издатель/аккаунт, отдельный автор текста может быть неизвестен.
    if quality_path.stem in ['livejournal', 'dzen']:
        fields.pop('author')
        fields['publisher'] = ['selected_publisher', 'publisher']
    missing = {}
    for name, candidates in fields.items():
        count = sum(quality_empty(quality_value(record, candidates)) for record in records)
        missing[name] = {
            'count': count,
            'percent': round(100 * count / len(records), 2) if records else None,
            'fields_used': candidates,
        }

    statuses = Counter()
    error_reasons = Counter()
    fetch_errors = 0
    empty_text = 0
    expected_without_full_text = 0
    for record in records:
        status = record.get('fetch_status')
        # 4PDA не записывает fetch_status: не выдаём отсутствие поля за успешную загрузку.
        statuses[str(status) if not quality_empty(status) else 'not_recorded'] += 1
        if status == 'error' or record.get('fetch_error'):
            fetch_errors += 1
            error_reasons[str(record.get('fetch_error') or 'fetch_status=error без указанной причины')] += 1
        if quality_empty(record.get('text')):
            empty_text += 1
            if quality_path.stem == 'dzen' and record.get('text_status') in ['summary_only', 'metadata_only']:
                expected_without_full_text += 1

    id_duplicates = quality_duplicates(quality_value(record, fields['id']) for record in records)
    url_duplicates = quality_duplicates(quality_value(record, fields['url']) for record in records)
    result = {
        'file': str(quality_path.relative_to(PROJECT)),
        'records_count': len(records),
        'duplicates_by_id': id_duplicates,
        'duplicates_by_url': url_duplicates,
        'missing_fields': missing,
        'fetch_status_counts': dict(statuses),
        'empty_text_count': empty_text,
        'expected_without_full_text': expected_without_full_text,
        'unexpected_empty_text_count': empty_text - expected_without_full_text,
        'fetch_error_count': fetch_errors,
        'fetch_error_reasons': dict(error_reasons),
        'read_error_count': len(read_errors),
        'read_errors': read_errors,
    }
    quality_report['sources'][quality_path.stem] = result
    print(f'{quality_path.name}: {len(records)} записей; '
          f'дубли ID/URL: {id_duplicates["extra_records"]}/{url_duplicates["extra_records"]}; '
          f'ошибки загрузки/чтения: {fetch_errors}/{len(read_errors)}')
    print(f'  Пустой текст: {empty_text}, ожидаемо без полного текста: {expected_without_full_text}')
    print('  Пропуски:', ', '.join(f'{name}={values["percent"]}%' for name, values in missing.items()))
    print('  Статусы:', dict(statuses))
    if error_reasons:
        print('  Причины ошибок:', dict(error_reasons))

quality_report['totals'] = {
    'files': len(quality_report['sources']),
    'records': sum(s['records_count'] for s in quality_report['sources'].values()),
    'fetch_errors': sum(s['fetch_error_count'] for s in quality_report['sources'].values()),
    'read_errors': sum(s['read_error_count'] for s in quality_report['sources'].values()),
    'unexpected_empty_text': sum(s['unexpected_empty_text_count'] for s in quality_report['sources'].values()),
}
quality_report_path = PROJECT / 'quality_report.json'
quality_temporary = quality_report_path.with_suffix('.json.tmp')
quality_temporary.write_text(json.dumps(quality_report, ensure_ascii=False, indent=2), encoding='utf-8')
quality_temporary.replace(quality_report_path)
print('Машиночитаемый отчёт:', quality_report_path)


4pda.jsonl: 5539 записей; дубли ID/URL: 0/0; ошибки загрузки/чтения: 0/0
  Пустой текст: 0, ожидаемо без полного текста: 0
  Пропуски: id=0.0%, url=0.0%, author=0.0%, date=0.0%, text=0.0%
  Статусы: {'not_recorded': 5539}
drive2.jsonl: 97 записей; дубли ID/URL: 0/0; ошибки загрузки/чтения: 0/0
  Пустой текст: 0, ожидаемо без полного текста: 0
  Пропуски: id=0.0%, url=0.0%, author=0.0%, date=0.0%, text=0.0%
  Статусы: {'ok': 97}
dzen.jsonl: 47 записей; дубли ID/URL: 0/0; ошибки загрузки/чтения: 0/0
  Пустой текст: 12, ожидаемо без полного текста: 12
  Пропуски: id=0.0%, url=0.0%, date=0.0%, text=25.53%, publisher=0.0%
  Статусы: {'ok': 47}
irecommend.jsonl: 394 записей; дубли ID/URL: 0/0; ошибки загрузки/чтения: 0/0
  Пустой текст: 0, ожидаемо без полного текста: 0
  Пропуски: id=0.0%, url=0.0%, author=0.0%, date=0.0%, text=0.0%
  Статусы: {'ok': 394}
livejournal.jsonl: 399 записей; дубли ID/URL: 0/0; ошибки загрузки/чтения: 0/0
  Пустой текст: 0, ожидаемо без полного текста: 0
  Пропус